## Import config

In [12]:
# Корень текущей ветки: config/srcc этой же ветки, без зависимости от cwd.
from pathlib import Path
import sys, os
_cwd = Path.cwd().resolve()
_expected = 'dynamic'
_candidates = [_cwd, _cwd / _expected, _cwd.parent / _expected]
PROJECT_ROOT = next((x for x in _candidates if (x / 'srcc' / 'nbc_project').is_dir() and x.name == _expected), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f'Запускайте ноутбук из папки {_expected} или её родителя')
os.chdir(PROJECT_ROOT)
SRC_PATH = PROJECT_ROOT / 'srcc'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))
print('PROJECT_ROOT:', PROJECT_ROOT)


PROJECT_ROOT: /home/jovyan/work/loyalty_handoff/dynamic


In [96]:
SRC_PATH

PosixPath('/home/jovyan/work/loyalty_handoff/dynamic/srcc')

In [14]:
import json
import pandas as pd
import numpy as np
from math import ceil

from rail_connectors import connect
import rail_connectors

from nbc_project.feature_lists import (
    load_feature_lists,
    get_feature_list,
    select_existing_columns,
)

FEATURE_LISTS = load_feature_lists()

# Декабрьский инференс активной базы
TARGET_MONTH = "2026-12-01"

BASE_VARIANT = "active_base"
MODEL_VARIANT = "dynamic"

# Уникальный постфикс для наших таблиц
postfix = "dec_active_base_dynamic_2026_rub_filtered"

# Октябрьские транзакции для декабрьского инференса
TRANSACTION_SOURCE = (
    "sandbox_crft.cb_charges_compact_nbc_monthly_202610"
)

# Эксперимент с обученными моделями
EXPERIMENT_NAME = "202610_dynamic_rub_filtered"

TRAIN_START = "2025-09-01"
TRAIN_END_EXCLUSIVE = "2026-10-01"
VALID_MONTH = "2026-10-01"
TEST_MONTH = None

EXPERIMENT_DIR = (
    PROJECT_ROOT / "experiment_outputs" / EXPERIMENT_NAME
)

matches = []

for config_path in EXPERIMENT_DIR.glob("*/run_config.json"):
    cfg = json.loads(config_path.read_text(encoding="utf-8"))

    if (
        cfg["train_start"] == TRAIN_START
        and cfg["train_end_exclusive"] == TRAIN_END_EXCLUSIVE
        and cfg["valid_month"] == VALID_MONTH
        and cfg["test_month"] == TEST_MONTH
    ):
        matches.append((config_path.parent, cfg))

assert len(matches) == 1, (
    f"Ожидался один запуск эксперимента, найдено: {len(matches)}"
)

RUN_DIR, run_config = matches[0]
RUN_TAG = run_config["run_tag"]

assert TARGET_MONTH == "2026-12-01"

print("TARGET_MONTH:", TARGET_MONTH)
print("postfix:", postfix)
print("TRANSACTION_SOURCE:", TRANSACTION_SOURCE)
print("RUN_DIR:", RUN_DIR)

TARGET_MONTH: 2026-12-01
postfix: dec_active_base_dynamic_2026_rub_filtered
TRANSACTION_SOURCE: sandbox_crft.cb_charges_compact_nbc_monthly_202610
RUN_DIR: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880


In [15]:
conn = connect(to="IMPALA")

with conn:
    connection_check = conn.fetch("""
        SELECT 1 AS connection_ok
    """)

print("Проверка подключения к Impala:")
display(connection_check)

Successfully login!
Проверка подключения к Impala:


,connection_ok
0,1


## Витрина клиентов для скоринга

In [16]:
# Исходная таблица с распределением клиентов по группам
BASE_GROUPS = "sandbox_crft.pia_base_groups"

# Наша выходная таблица для декабрьского инференса
CLIENTS_FOR_LOYALTY = (
    f"sandbox_crft.mae_client_for_loyalty_{postfix}"
)

# Проверяем активную базу до создания каких-либо таблиц
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT cdi_id) AS unique_clients,
    SUM(CASE WHEN cdi_id IS NULL THEN 1 ELSE 0 END) AS null_clients
FROM {BASE_GROUPS}
WHERE pilot_group = 'active_base'
"""

with conn:
    base_stats = conn.fetch(query)

base_stats["duplicate_rows"] = (
    base_stats["total_rows"]
    - base_stats["unique_clients"]
    - base_stats["null_clients"]
)

print("Источник:", BASE_GROUPS)
print("Группа: active_base")
print("Выходная таблица:", CLIENTS_FOR_LOYALTY)

display(base_stats)

Источник: sandbox_crft.pia_base_groups
Группа: active_base
Выходная таблица: sandbox_crft.mae_client_for_loyalty_dec_active_base_dynamic_2026_rub_filtered


/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,total_rows,unique_clients,null_clients,duplicate_rows
0,696615,696615,0.0,0.0


In [17]:
with conn:
    conn.execute(f'''drop table if exists {CLIENTS_FOR_LOYALTY}''')

In [18]:
# with conn:
#     conn.execute(f'''
#     CREATE TABLE {CLIENTS_FOR_LOYALTY}
#     TBLPROPERTIES('transactional' = 'false')
#     AS with a as (
#         select
#             customer_id,
#             abs(sum(charge_rub_amt))/count(distinct TRUNC(tt.trans_dt, 'mm')) as charge_rub_amt
#         from dm_dcp_fl.tfct_transaction tt
#         left join dm_dcp_fl.dim_mcc mcc
#             on tt.mcc_id = mcc.mcc_id 
#         where tt.mcc_id != -99
#             and customer_id is not null
#             and trans_dt >= '{start_date}' and trans_dt < '{end_date}'
#         group by customer_id
#         ),
#         b as (
#             select 
#                 a.customer_id, c.cdi_party_id, count(*)
#             from a
#             left join dm_dcp_fl.ref_customer c
#                 on c.customer_id = a.customer_id
#             where cdi_party_id is not null
#             group by a.customer_id, c.cdi_party_id
#         )
#         select distinct cast('{TARGET_MONTH}' as date) as period_start_date, cdi_party_id as cdi_id
#         from b
#         limit 200
#     ''')

In [19]:
with conn:
    conn.execute(f'''
    CREATE TABLE {CLIENTS_FOR_LOYALTY}
    TBLPROPERTIES('transactional' = 'false')
    AS
        select
            cast('{TARGET_MONTH}' as date) as period_start_date
            , cast(cdi_id as bigint) as cdi_id
        from {BASE_GROUPS}
        where pilot_group = 'active_base'
    ''')

In [20]:
with conn:
    d = conn.fetch(
        f'''
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT cdi_id) AS unique_clients
        FROM {CLIENTS_FOR_LOYALTY}
        '''
    )

    df_all = conn.fetch(
        f'SELECT * FROM {CLIENTS_FOR_LOYALTY} LIMIT 5'
    )

display(d)
display(df_all)

,total_rows,unique_clients
0,696615,696615


,period_start_date,cdi_id
0,2026-12-01,1031529347
1,2026-12-01,31941817572
2,2026-12-01,43878932960
3,2026-12-01,38686954065
4,2026-12-01,1294708945


In [ ]:
with conn:
    d = conn.fetch(f'''select count(*), count(distinct(cdi_id)) from {CLIENTS_FOR_LOYALTY}''')
    df_all = conn.fetch(f'''select * from {CLIENTS_FOR_LOYALTY} limit 5''')
d

In [ ]:
df_all

## Витрина сценариев для клиентов

In [23]:
SEND_TO_DBO = f"sandbox_crft.send_to_dbo_{postfix}"

# SOURCE_TABLE = "sandbox_crft.mae_nbc_optimizer_scores_fm_bin_june_cat_id"
# SOURCE_TABLE = "sandbox_crft.mae_nbc_optimizer_scores_fm_bin_august_2026_catid"

# Внешний источник уже содержит готовые 10 строк на сценарий:
# cdi_id, scenario_id, coefficient, category_id, category_percent
# EXTERNAL_SOURCE_TABLE = "sandbox_crft.external_client_scenarios"

# Пул категорий для random:
# category_id
CATEGORY_POOL_TABLE = "sandbox_crft.mae_category_pool"

# COST_SCENARIOS_TABLE = "sandbox_crft.send_to_dbo_cost_scenarios"
# UPLIFT_SCENARIOS_TABLE = "sandbox_crft.send_to_dbo_uplift_scenarios"
EXTERNAL_SCENARIOS_TABLE = "sandbox_crft.send_to_dbo_external_scenarios_202612"

RANDOM_SCENARIOS_TABLE = (
    f"sandbox_crft.send_to_dbo_random_scenarios_{postfix}"
)

RANDOM_TMP_PREFIX = (
    f"sandbox_crft.send_to_dbo_random_{postfix}"
)

SOURCE_CLIENT_COL = "cdi_id"
SOURCE_CATEGORY_COL = "category_id"

# ALL_PURCHASES_CATEGORY_ID = "d40cd3f8-5b8c-426f-84a4-46c848f7b901"
ALL_PURCHASES_CATEGORY_ID = "9a038f75-65c1-43b5-95bc-62967a8d1100"

CAPPED_CATEGORIES = (
    "9a038f75-65c1-43b5-95bc-62967a8d1133",
    "9a038f75-65c1-43b5-95bc-62967a8d1134",
)

CAPPED_CATEGORIES_SQL = ", ".join(f"'{x}'" for x in CAPPED_CATEGORIES)
# Имя декабрьского внешнего источника выведено по ноябрьскому шаблону; наличие строк проверяем ДО CREATE/DROP.


In [24]:
# Проверяем, какие источники сценариев существуют в Impala

patterns = [
    "send_to_dbo_external*",
    "external_client_scenarios*",
    "send_to_dbo_dec_active_base*",
]

for pattern in patterns:
    print(f"\nПоиск таблиц: {pattern}")

    with conn:
        tables = conn.fetch(
            f"SHOW TABLES IN sandbox_crft LIKE '{pattern}'"
        )

    display(tables)

print("\nПредполагаемый внешний источник:")
print(EXTERNAL_SCENARIOS_TABLE)

print("\nНаша будущая random-таблица:")
print(RANDOM_SCENARIOS_TABLE)

print("\nНаша итоговая таблица:")
print(SEND_TO_DBO)


Поиск таблиц: send_to_dbo_external*


,name
0,send_to_dbo_external_scenarios
1,send_to_dbo_external_scenarios_202608
2,send_to_dbo_external_scenarios_202609
3,send_to_dbo_external_scenarios_202610
4,send_to_dbo_external_scenarios_202611
5,send_to_dbo_external_scenarios_202612



Поиск таблиц: external_client_scenarios*


,name
0,external_client_scenarios



Поиск таблиц: send_to_dbo_dec_active_base*


,name



Предполагаемый внешний источник:
sandbox_crft.send_to_dbo_external_scenarios_202612

Наша будущая random-таблица:
sandbox_crft.send_to_dbo_random_scenarios_dec_active_base_dynamic_2026_rub_filtered

Наша итоговая таблица:
sandbox_crft.send_to_dbo_dec_active_base_dynamic_2026_rub_filtered


In [25]:
# Проверяем внешний источник декабрьских сценариев

print("Источник:", EXTERNAL_SCENARIOS_TABLE)
print("Период:", TARGET_MONTH)

with conn:
    # Структура таблицы
    external_schema = conn.fetch(
        f"DESCRIBE {EXTERNAL_SCENARIOS_TABLE}"
    )

    # Пример декабрьских сценариев
    external_sample = conn.fetch(f"""
        SELECT *
        FROM {EXTERNAL_SCENARIOS_TABLE}
        WHERE CAST(period_start_date AS DATE) =
              CAST('{TARGET_MONTH}' AS DATE)
        LIMIT 20
    """)

print("\nСтруктура таблицы:")
display(external_schema)

print("\nПример декабрьских сценариев:")
display(external_sample)

assert not external_sample.empty, (
    f"Нет сценариев за {TARGET_MONTH}"
)

Источник: sandbox_crft.send_to_dbo_external_scenarios_202612
Период: 2026-12-01

Структура таблицы:


/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,name,type,comment
0,cdi_id,string,
1,period_start_date,date,
2,thematic_id,string,
3,coefficient,double,
4,scenario_num,int,
5,category_id,string,
6,category_percent,bigint,



Пример декабрьских сценариев:


,cdi_id,period_start_date,thematic_id,coefficient,scenario_num,category_id,category_percent
0,56638423,2026-12-01,d40cd3f8-5b8c-426f-84a4-46c848f7b901,0.3,1,9a038f75-65c1-43b5-95bc-62967a8d1100,1
1,56638423,2026-12-01,d40cd3f8-5b8c-426f-84a4-46c848f7b901,0.3,1,9a038f75-65c1-43b5-95bc-62967a8d1101,15
2,56638423,2026-12-01,d40cd3f8-5b8c-426f-84a4-46c848f7b901,0.3,1,9a038f75-65c1-43b5-95bc-62967a8d1108,11
3,56638423,2026-12-01,d40cd3f8-5b8c-426f-84a4-46c848f7b901,0.3,1,9a038f75-65c1-43b5-95bc-62967a8d1109,5
4,56638423,2026-12-01,d40cd3f8-5b8c-426f-84a4-46c848f7b901,0.3,1,9a038f75-65c1-43b5-95bc-62967a8d1121,5
5,56638423,2026-12-01,d40cd3f8-5b8c-426f-84a4-46c848f7b901,0.3,1,9a038f75-65c1-43b5-95bc-62967a8d1111,5
6,56638423,2026-12-01,d40cd3f8-5b8c-426f-84a4-46c848f7b901,0.3,1,9a038f75-65c1-43b5-95bc-62967a8d1125,5
7,56638423,2026-12-01,d40cd3f8-5b8c-426f-84a4-46c848f7b901,0.3,1,9a038f75-65c1-43b5-95bc-62967a8d1130,5
8,56638423,2026-12-01,d40cd3f8-5b8c-426f-84a4-46c848f7b901,0.3,1,9a038f75-65c1-43b5-95bc-62967a8d1106,5
9,56638423,2026-12-01,d40cd3f8-5b8c-426f-84a4-46c848f7b901,0.3,1,9a038f75-65c1-43b5-95bc-62967a8d1124,5


### build_random_topk_scenario_table

#### Новая

In [27]:
def build_random_topk_scenario_table(
    conn,
    output_table: str,
    tmp_prefix: str,
    top_k_random_scenarios: int = 20,
):
    """
    Генерирует случайные сценарии для клиентов.

    Один сценарий:
        1 категория "Все покупки" — 1%
        1 категория — 15%
        1 категория — 11%
        7 категорий — 5%

    Правила:
        - 10 уникальных категорий.
        - Исключённые категории не используются.
        - HIGH: только 11% из повышенных ставок.
        - MEDIUM: 11% или 15%, но не две MEDIUM одновременно.
        - LOW: 11% или 15%.
        - NO_WEIGHT: только 5%.
        - CAPPED_CATEGORIES: только 5%.

    Вместо CROSS JOIN всех клиентов со всеми категориями
    создаём небольшую библиотеку случайных шаблонов.

    Затем каждому клиенту случайно назначаем различные шаблоны.
    """

    # Простое число для равномерного распределения шаблонов.
    # При шаге от 1 до 9972 номера шаблонов не повторяются
    # в пределах первых 9973 назначений.
    template_count = 9973

    if not 1 <= top_k_random_scenarios <= template_count:
        raise ValueError(
            "top_k_random_scenarios должен быть от 1 "
            f"до {template_count}"
        )

    # ----------------------------------------------------------
    # 1. Исключённые категории
    # ----------------------------------------------------------

    excluded_categories = (
        "9a038f75-65c1-43b5-95bc-62967a8d1137",
        "9a038f75-65c1-43b5-95bc-62967a8d1138",
        "9a038f75-65c1-43b5-95bc-62967a8d1135",
        "9a038f75-65c1-43b5-95bc-62967a8d1139",
        "9a038f75-65c1-43b5-95bc-62967a8d1140",
        "9a038f75-65c1-43b5-95bc-62967a8d1114",
        "9a038f75-65c1-43b5-95bc-62967a8d1117",
        "9a038f75-65c1-43b5-95bc-62967a8d1119",
        "9a038f75-65c1-43b5-95bc-62967a8d1132",
    )

    excluded_sql = ", ".join(
        f"'{category_id}'"
        for category_id in (
            *excluded_categories,
            ALL_PURCHASES_CATEGORY_ID,
        )
    )

    capped_sql = ", ".join(
        f"'{category_id}'"
        for category_id in CAPPED_CATEGORIES
    )

    # ----------------------------------------------------------
    # 2. Имена временных таблиц
    # ----------------------------------------------------------

    pool_table = f"{tmp_prefix}_pool"
    pairs_table = f"{tmp_prefix}_pairs"

    template_seeds = f"{tmp_prefix}_template_seeds"
    template_pairs = f"{tmp_prefix}_template_pairs"

    template_candidates = f"{tmp_prefix}_template_candidates"
    template_ranked = f"{tmp_prefix}_template_ranked"
    template_rows = f"{tmp_prefix}_template_rows"

    client_seeds = f"{tmp_prefix}_client_seeds"

    def recreate_table(table_name, select_sql):
        print(f"[random] Создаю {table_name}")

        with conn:
            conn.execute(
                f"DROP TABLE IF EXISTS {table_name}"
            )

            conn.execute(f"""
                CREATE TABLE {table_name}
                STORED AS PARQUET
                AS
                {select_sql}
            """)

    # ----------------------------------------------------------
    # 3. Разрешённый пул категорий + POS-типы
    # ----------------------------------------------------------

    recreate_table(
        pool_table,
        f"""
        WITH global_weights AS (
            SELECT
                CAST(category_id AS STRING) AS category_id,
                MAX(`var`) AS global_pos_weight
            FROM sandbox_crft.pia_loyal_group_test_global_weights
            GROUP BY CAST(category_id AS STRING)
        )

        SELECT DISTINCT
            CAST(c.category_id AS STRING) AS category_id,

            CASE
                WHEN g.global_pos_weight >= 0.03 THEN 'high'
                WHEN g.global_pos_weight >= 0.01 THEN 'medium'
                WHEN g.global_pos_weight IS NOT NULL THEN 'low'
                ELSE 'no_weight'
            END AS pos_type,

            CASE
                WHEN CAST(c.category_id AS STRING)
                     IN ({capped_sql})
                THEN 0
                ELSE 1
            END AS can_promote

        FROM sandbox_crft.pia_loyal_group_test_o_categories_actual c

        LEFT JOIN global_weights g
            ON CAST(c.category_id AS STRING) = g.category_id

        WHERE CAST(c.category_id AS STRING)
              NOT IN ({excluded_sql})
        """
    )

    with conn:
        pool_stats = conn.fetch(f"""
            SELECT
                pos_type,
                COUNT(*) AS n_categories
            FROM {pool_table}
            GROUP BY pos_type
            ORDER BY pos_type
        """)

    print("\nРазрешённые категории:")
    print(pool_stats.to_string(index=False))

    n_categories = int(pool_stats["n_categories"].sum())

    if n_categories < 9:
        raise ValueError(
            "Недостаточно категорий для формирования сценария: "
            f"{n_categories}"
        )

    # ----------------------------------------------------------
    # 4. Все допустимые пары 11% + 15%
    # ----------------------------------------------------------

    recreate_table(
        pairs_table,
        f"""
        SELECT
            ROW_NUMBER() OVER (
                ORDER BY
                    c11.category_id,
                    c15.category_id
            ) AS pair_id,

            c11.category_id AS category_11,
            c15.category_id AS category_15

        FROM {pool_table} c11

        JOIN {pool_table} c15
            ON c11.category_id != c15.category_id

        WHERE
            c11.can_promote = 1
            AND c15.can_promote = 1

            AND c11.pos_type IN ('high', 'medium', 'low')
            AND c15.pos_type IN ('medium', 'low')

            AND NOT (
                c11.pos_type = 'medium'
                AND c15.pos_type = 'medium'
            )
        """
    )

    with conn:
        pair_stats = conn.fetch(f"""
            SELECT COUNT(*) AS n_pairs
            FROM {pairs_table}
        """)

    n_pairs = int(pair_stats.iloc[0]["n_pairs"])

    print(f"\nДопустимых пар 11% + 15%: {n_pairs}")

    if n_pairs == 0:
        raise ValueError("Нет допустимых пар повышенных категорий")

    # ----------------------------------------------------------
    # 5. Создаём номера случайных шаблонов
    # ----------------------------------------------------------

    digits_sql = "\nUNION ALL\n".join(
        f"SELECT {i} AS d"
        for i in range(10)
    )

    recreate_table(
        template_seeds,
        f"""
        WITH digits AS (
            {digits_sql}
        )

        SELECT
            CAST(
                1
                + d0.d
                + 10 * d1.d
                + 100 * d2.d
                + 1000 * d3.d
                AS INT
            ) AS template_id,

            RAND() AS pair_random

        FROM digits d0
        CROSS JOIN digits d1
        CROSS JOIN digits d2
        CROSS JOIN digits d3

        WHERE
            d0.d
            + 10 * d1.d
            + 100 * d2.d
            + 1000 * d3.d
            < {template_count}
        """
    )

    # ----------------------------------------------------------
    # 6. Случайно выбираем пару для каждого шаблона
    # ----------------------------------------------------------

    recreate_table(
        template_pairs,
        f"""
        SELECT
            t.template_id,
            p.category_11,
            p.category_15

        FROM {template_seeds} t

        CROSS JOIN (
            SELECT COUNT(*) AS n_pairs
            FROM {pairs_table}
        ) pc

        JOIN {pairs_table} p
            ON p.pair_id = (
                CAST(
                    FLOOR(t.pair_random * pc.n_pairs)
                    AS BIGINT
                ) + 1
            )
        """
    )

    # ----------------------------------------------------------
    # 7. Случайные кандидаты на оставшиеся 7 мест
    # ----------------------------------------------------------

    recreate_table(
        template_candidates,
        f"""
        SELECT
            t.template_id,
            c.category_id,
            RAND() AS random_score

        FROM {template_pairs} t

        CROSS JOIN {pool_table} c

        WHERE c.category_id NOT IN (
            t.category_11,
            t.category_15
        )
        """
    )

    recreate_table(
        template_ranked,
        f"""
        SELECT
            template_id,
            category_id,

            ROW_NUMBER() OVER (
                PARTITION BY template_id
                ORDER BY random_score, category_id
            ) AS category_rank

        FROM {template_candidates}
        """
    )

    # ----------------------------------------------------------
    # 8. Собираем 10 категорий каждого шаблона
    # ----------------------------------------------------------

    recreate_table(
        template_rows,
        f"""
        SELECT
            template_id,
            CAST(1 AS INT) AS category_rank,
            CAST('{ALL_PURCHASES_CATEGORY_ID}' AS STRING)
                AS category_id,
            CAST(1 AS INT) AS category_percent,
            'mandatory_all_purchases' AS category_role
        FROM {template_pairs}

        UNION ALL

        SELECT
            template_id,
            CAST(2 AS INT) AS category_rank,
            category_15 AS category_id,
            CAST(15 AS INT) AS category_percent,
            'variable_percent' AS category_role
        FROM {template_pairs}

        UNION ALL

        SELECT
            template_id,
            CAST(3 AS INT) AS category_rank,
            category_11 AS category_id,
            CAST(11 AS INT) AS category_percent,
            'variable_percent' AS category_role
        FROM {template_pairs}

        UNION ALL

        SELECT
            template_id,
            CAST(category_rank + 3 AS INT) AS category_rank,
            category_id,
            CAST(5 AS INT) AS category_percent,
            'fixed_5' AS category_role
        FROM {template_ranked}
        WHERE category_rank <= 7
        """
    )

    # ----------------------------------------------------------
    # 9. Проверяем библиотеку сценариев
    # ----------------------------------------------------------

    with conn:
        check_templates = conn.fetch(f"""
            WITH template_stats AS (
                SELECT
                    template_id,
                    COUNT(*) AS n,
                    COUNT(DISTINCT category_id) AS n_unique,

                    SUM(
                        CASE WHEN category_percent = 1
                        THEN 1 ELSE 0 END
                    ) AS n_1,

                    SUM(
                        CASE WHEN category_percent = 5
                        THEN 1 ELSE 0 END
                    ) AS n_5,

                    SUM(
                        CASE WHEN category_percent = 11
                        THEN 1 ELSE 0 END
                    ) AS n_11,

                    SUM(
                        CASE WHEN category_percent = 15
                        THEN 1 ELSE 0 END
                    ) AS n_15

                FROM {template_rows}
                GROUP BY template_id
            )

            SELECT
                COUNT(*) AS n_templates,

                SUM(
                    CASE
                        WHEN n != 10
                          OR n_unique != 10
                          OR n_1 != 1
                          OR n_5 != 7
                          OR n_11 != 1
                          OR n_15 != 1
                        THEN 1 ELSE 0
                    END
                ) AS invalid_templates

            FROM template_stats
        """)

    print("\nПроверка шаблонов:")
    print(check_templates.to_string(index=False))

    actual_templates = int(
        check_templates.iloc[0]["n_templates"]
    )

    invalid_templates = int(
        check_templates.iloc[0]["invalid_templates"]
    )

    if actual_templates != template_count or invalid_templates != 0:
        raise ValueError(
            "Ошибка формирования случайных шаблонов"
        )

    # ----------------------------------------------------------
    # 10. Случайные начальные позиции для клиентов
    # ----------------------------------------------------------

    recreate_table(
        client_seeds,
        f"""
        SELECT
            CAST(cdi_id AS STRING) AS cdi_id,
            CAST(period_start_date AS DATE) AS period_start_date,

            CAST(
                FLOOR(RAND() * {template_count})
                AS INT
            ) AS start_index,

            CAST(
                FLOOR(RAND() * {template_count - 1}) + 1
                AS INT
            ) AS step_size

        FROM {CLIENTS_FOR_LOYALTY}
        """
    )

    # ----------------------------------------------------------
    # 11. Назначаем клиентам случайные шаблоны
    # ----------------------------------------------------------

    scenario_nums_sql = "\nUNION ALL\n".join(
        f"SELECT {i} AS scenario_num"
        for i in range(1, top_k_random_scenarios + 1)
    )

    recreate_table(
        output_table,
        f"""
        WITH scenario_nums AS (
            {scenario_nums_sql}
        )

        SELECT
            c.cdi_id,
            c.period_start_date,

            CONCAT(
                'client_random_topn_',
                CAST(s.scenario_num AS STRING)
            ) AS offer_scenario_id,

            'random' AS offer_source,

            CAST(s.scenario_num AS INT) AS scenario_num,

            t.category_rank,
            t.category_id,
            t.category_percent,
            t.category_role,

            CAST(NULL AS DOUBLE) AS source_score,
            CAST(1 AS INT) AS is_random_fallback,

            CURRENT_TIMESTAMP() AS created_at

        FROM {client_seeds} c

        CROSS JOIN scenario_nums s

        JOIN {template_rows} t
            ON t.template_id = CAST(
                (
                    (
                        c.start_index
                        + (s.scenario_num - 1) * c.step_size
                    ) % {template_count}
                ) + 1
                AS INT
            )
        """
    )

    print(f"\n[random] Готово: {output_table}")

    return {
        "output_table": output_table,
        "pool_table": pool_table,
        "pairs_table": pairs_table,
        "template_table": template_rows,
        "client_seeds_table": client_seeds,
        "template_count": template_count,
        "scenarios_per_client": top_k_random_scenarios,
    }

### Вызов функций

In [28]:
%%time

# Отдельные таблицы для нашего декабрьского запуска
RANDOM_SCENARIOS_TABLE = (
    f"sandbox_crft.send_to_dbo_random_scenarios_{postfix}"
)

RANDOM_TMP_PREFIX = (
    f"sandbox_crft.send_to_dbo_random_{postfix}"
)

# Генерируем 20 случайных сценариев на каждого клиента
random_result = build_random_topk_scenario_table(
    conn=conn,
    output_table=RANDOM_SCENARIOS_TABLE,
    tmp_prefix=RANDOM_TMP_PREFIX,
    top_k_random_scenarios=20,
)

print("\nRandom-сценарии сформированы")
print("Таблица:", RANDOM_SCENARIOS_TABLE)
print("Количество шаблонов:", random_result["template_count"])
print("Сценариев на клиента:", random_result["scenarios_per_client"])

[random] Создаю sandbox_crft.send_to_dbo_random_dec_active_base_dynamic_2026_rub_filtered_pool

Разрешённые категории:
pos_type n_categories
    high           10
     low           14
  medium            7
[random] Создаю sandbox_crft.send_to_dbo_random_dec_active_base_dynamic_2026_rub_filtered_pairs

Допустимых пар 11% + 15%: 546
[random] Создаю sandbox_crft.send_to_dbo_random_dec_active_base_dynamic_2026_rub_filtered_template_seeds
[random] Создаю sandbox_crft.send_to_dbo_random_dec_active_base_dynamic_2026_rub_filtered_template_pairs
[random] Создаю sandbox_crft.send_to_dbo_random_dec_active_base_dynamic_2026_rub_filtered_template_candidates
[random] Создаю sandbox_crft.send_to_dbo_random_dec_active_base_dynamic_2026_rub_filtered_template_ranked
[random] Создаю sandbox_crft.send_to_dbo_random_dec_active_base_dynamic_2026_rub_filtered_template_rows

Проверка шаблонов:
n_templates invalid_templates
       9973                 0
[random] Создаю sandbox_crft.send_to_dbo_random_dec_acti

In [34]:
%%time

# Итоговая витрина: внешний алгоритм + random
# Источник Current больше не используем.

print("Внешний источник:", EXTERNAL_SCENARIOS_TABLE)
print("Random:", RANDOM_SCENARIOS_TABLE)
print("Результат:", SEND_TO_DBO)

query = f"""
CREATE TABLE {SEND_TO_DBO}
STORED AS PARQUET
AS

-- 1. Сценарии внешнего алгоритма
SELECT
    CAST(e.cdi_id AS STRING) AS cdi_id,
    CAST(e.period_start_date AS DATE) AS period_start_date,

    CONCAT(
        'fix_',
        CAST(
            CAST(
                ROUND(e.coefficient, 2)
                AS DECIMAL(10, 2)
            )
            AS STRING
        ),
        '_',
        CAST(e.scenario_num AS STRING)
    ) AS offer_scenario_id,

    'fixed' AS offer_source,

    CAST(e.category_id AS STRING) AS category_id,
    CAST(e.category_percent AS BIGINT) AS category_percent

FROM {EXTERNAL_SCENARIOS_TABLE} e

INNER JOIN {CLIENTS_FOR_LOYALTY} b
    ON CAST(e.cdi_id AS STRING) = CAST(b.cdi_id AS STRING)

WHERE CAST(e.period_start_date AS DATE) =
      CAST('{TARGET_MONTH}' AS DATE)

UNION ALL

-- 2. Случайные сценарии
SELECT
    CAST(r.cdi_id AS STRING) AS cdi_id,
    CAST(r.period_start_date AS DATE) AS period_start_date,
    CAST(r.offer_scenario_id AS STRING) AS offer_scenario_id,
    'random' AS offer_source,
    CAST(r.category_id AS STRING) AS category_id,
    CAST(r.category_percent AS BIGINT) AS category_percent

FROM {RANDOM_SCENARIOS_TABLE} r

WHERE CAST(r.period_start_date AS DATE) =
      CAST('{TARGET_MONTH}' AS DATE)
"""

with conn:
    conn.execute(f"DROP TABLE IF EXISTS {SEND_TO_DBO}")
    conn.execute(query)

print("\nИтоговая витрина успешно создана:")
print(SEND_TO_DBO)

Внешний источник: sandbox_crft.send_to_dbo_external_scenarios_202612
Random: sandbox_crft.send_to_dbo_random_scenarios_dec_active_base_dynamic_2026_rub_filtered
Результат: sandbox_crft.send_to_dbo_dec_active_base_dynamic_2026_rub_filtered

Итоговая витрина успешно создана:
sandbox_crft.send_to_dbo_dec_active_base_dynamic_2026_rub_filtered
CPU times: user 396 ms, sys: 885 ms, total: 1.28 s
Wall time: 1min 41s


In [30]:
%%time

# Проверяем итоговое количество сценариев
# отдельно по внешнему алгоритму и random

query = f"""
SELECT
    offer_source,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT cdi_id) AS unique_clients,
    COUNT(DISTINCT offer_scenario_id) AS unique_scenario_ids
FROM {SEND_TO_DBO}
WHERE period_start_date = CAST('{TARGET_MONTH}' AS DATE)
GROUP BY offer_source
ORDER BY offer_source
"""

with conn:
    scenario_summary = conn.fetch(query)

display(scenario_summary)

# Проверяем ожидаемый объём random
expected_clients = 696_615
expected_random_per_client = 20
expected_categories_per_scenario = 10

expected_random_rows = (
    expected_clients
    * expected_random_per_client
    * expected_categories_per_scenario
)

random_stats = scenario_summary[
    scenario_summary["offer_source"] == "random"
]

assert len(random_stats) == 1, "Отсутствуют random-сценарии"

assert int(random_stats.iloc[0]["total_rows"]) == expected_random_rows, (
    "Количество строк random-сценариев отличается от ожидаемого"
)

assert int(random_stats.iloc[0]["unique_clients"]) == expected_clients, (
    "Random-сценарии сформированы не для всех клиентов"
)

print("\nПроверка random пройдена")
print("Ожидаемых строк:", expected_random_rows)
print("Фактических строк:", int(random_stats.iloc[0]["total_rows"]))

,offer_source,total_rows,unique_clients,unique_scenario_ids
0,fixed,243815250,696615,65
1,random,139323000,696615,20



Проверка random пройдена
Ожидаемых строк: 139323000
Фактических строк: 139323000
CPU times: user 156 ms, sys: 187 ms, total: 343 ms
Wall time: 19.9 s


In [35]:
%%time

# Проверяем итоговое количество сценариев
# отдельно по внешнему алгоритму и random

query = f"""
SELECT
    *
FROM {SEND_TO_DBO}
WHERE period_start_date = CAST('{TARGET_MONTH}' AS DATE)
limit 5
"""

with conn:
    temp = conn.fetch(query)

temp

CPU times: user 42.6 ms, sys: 14 ms, total: 56.7 ms
Wall time: 1.24 s


,cdi_id,period_start_date,offer_scenario_id,offer_source,category_id,category_percent
0,56638812,2026-12-01,fix_0.30_1,fixed,9a038f75-65c1-43b5-95bc-62967a8d1100,1
1,56638812,2026-12-01,fix_0.30_1,fixed,9a038f75-65c1-43b5-95bc-62967a8d1108,15
2,56638812,2026-12-01,fix_0.30_1,fixed,9a038f75-65c1-43b5-95bc-62967a8d1121,11
3,56638812,2026-12-01,fix_0.30_1,fixed,9a038f75-65c1-43b5-95bc-62967a8d1101,5
4,56638812,2026-12-01,fix_0.30_1,fixed,9a038f75-65c1-43b5-95bc-62967a8d1125,5


In [ ]:
1

## Сборка фичей

In [36]:
cb_base = f"sandbox_crft.cb_base_client_month_{postfix}"
cb_base_scen = f"sandbox_crft.cb_base_scenario_client_month_{postfix}"
# cb_charges_compact = f"sandbox_crft.cb_charges_compact_04_06_cashback_total_pred"
# cb_charges_compact = f"sandbox_crft.cb_charges_compact_june_scenarios_train_pred"
# cb_charges_compact = f"sandbox_crft.cb_charges_compact_august_oot_train_pred"
cb_charges_compact = TRANSACTION_SOURCE
cb_category_spend_wide = f"sandbox_crft.cb_category_spend_wide_{postfix}"
cb_offered_percent_wide = f"sandbox_crft.cb_offered_percent_wide_{postfix}"
cb_selected_history_wide = f"sandbox_crft.cb_selected_history_wide_{postfix}"
cb_acrm_monthly = f"sandbox_crft.cb_acrm_monthly_{postfix}"
cb_acrm_targets = f"sandbox_crft.cb_acrm_targets_{postfix}"
cb_acrm_lags = f"sandbox_crft.cb_acrm_lags_{postfix}"
cb_activity_features = f"sandbox_crft.cb_activity_features_{postfix}"
cb_potential_features = f"sandbox_crft.cb_potential_features_{postfix}"
cb_share_features = f"sandbox_crft.cb_share_features_{postfix}"
cb_dataset = f"sandbox_crft.cb_client_month_dataset_{postfix}"

LOYALTY_CLIENT_CATEGORIES = 'sandbox_crft.mae_loyalty_client_categories'
send_to_dbo = SEND_TO_DBO
# send_to_dbo = 'sandbox_crft.pia_final_loyal_202608'

def run_sql(conn, queries, compute_stats_table=None, preview_table=None, limit=5):
    with conn:
        for query in queries:
            conn.execute(query)

        if compute_stats_table is not None:
            conn.execute(f"compute stats {compute_stats_table}")

        if preview_table is not None:
            return conn.fetch(f"""
                select *
                from {preview_table}
                limit {limit}
            """)


def category_suffix(category_id: str) -> str:
    return str(category_id).replace("-", "")[-4:]


start_month = '2026-03-01'
end_month = '2026-12-01'
assert cb_charges_compact == TRANSACTION_SOURCE

# Общий октябрьский источник для декабрьской истории: проверяем доступные месяцы.
with conn:
    _tx_months = conn.fetch(f"SELECT TRUNC(CAST(month_date AS DATE), 'MM') AS m, COUNT(*) AS n FROM {cb_charges_compact} WHERE CAST(month_date AS DATE) >= '2026-05-01' AND CAST(month_date AS DATE) < '2026-11-01' GROUP BY TRUNC(CAST(month_date AS DATE), 'MM') ORDER BY m")
_expected_tx = {'2026-05','2026-06','2026-07','2026-08','2026-09','2026-10'}
_actual_tx = set(pd.to_datetime(_tx_months['m']).dt.strftime('%Y-%m')) if not _tx_months.empty else set()
assert _expected_tx <= _actual_tx, f'Не хватает месяцев в cb_charges_compact: {_expected_tx-_actual_tx}'
print('Месяцы истории:', sorted(_actual_tx))
print('ВНИМАНИЕ: сверить покрытие клиентов декабрьской базы этой витриной; она создана на октябрьской обучающей базе!')


Месяцы истории: ['2026-05', '2026-06', '2026-07', '2026-08', '2026-09', '2026-10']
ВНИМАНИЕ: сверить покрытие клиентов декабрьской базы этой витриной; она создана на октябрьской обучающей базе!


In [ ]:
send_to_dbo

### 1. Базовая таблица клиент-месяц-сценарий

In [37]:
%%time
drop = f"""
drop table if exists {cb_base}
"""

create = f"""
create table {cb_base} stored as parquet as
select distinct
    cast(cdi_id as bigint) as cdi_id,
    cast(period_start_date as timestamp) as forecast_month
from {send_to_dbo}
where period_start_date >= cast('{start_month}' as timestamp)
"""

with conn:
    for query in [drop, create]:
        conn.execute(query)
        
    df = conn.fetch(f"""select * from {cb_base} limit 5""")
df

CPU times: user 110 ms, sys: 88.5 ms, total: 198 ms
Wall time: 8.7 s


,cdi_id,forecast_month
0,1384426493,2026-12-01 00:00:00
1,24870434725,2026-12-01 00:00:00
2,45748194010,2026-12-01 00:00:00
3,45721752723,2026-12-01 00:00:00
4,1561788260,2026-12-01 00:00:00


In [38]:
# Диагностика покрытия декабрьской аудитории октябрьской обучающей транзакционной витриной.
# Отсутствие транзакций не равно отсутствию клиента: сравнивайте с нормальной долей активных клиентов.
with conn:
    _cov = conn.fetch(f"""
        SELECT COUNT(DISTINCT b.cdi_id) AS scoring_clients,
               COUNT(DISTINCT t.cdi_id) AS clients_with_history
        FROM {cb_base} b
        LEFT JOIN {cb_charges_compact} t
          ON b.cdi_id = t.cdi_id
         AND CAST(t.month_date AS DATE) >= '2026-05-01'
         AND CAST(t.month_date AS DATE) < '2026-11-01'
    """)
print('Сравните покрытие транзакционной историей с ожидаемой долей активных клиентов:', _cov)


Сравните покрытие транзакционной историей с ожидаемой долей активных клиентов:   scoring_clients clients_with_history
0          696615               578916


In [39]:
%%time
drop = f"""
drop table if exists {cb_base_scen}
"""

create = f"""
create table {cb_base_scen} stored as parquet as
select distinct
    cast(cdi_id as bigint) as cdi_id,
    cast(period_start_date as timestamp) as forecast_month,
    offer_scenario_id
from {send_to_dbo}
where period_start_date >= cast('{start_month}' as timestamp)
"""

with conn:
    for query in [drop, create]:
        conn.execute(query)
        
    df = conn.fetch(f"""select * from {cb_base_scen} limit 5""")
df

CPU times: user 123 ms, sys: 122 ms, total: 244 ms
Wall time: 11.8 s


,cdi_id,forecast_month,offer_scenario_id
0,1561539436,2026-12-01 00:00:00,fix_1.00_1
1,36340609205,2026-12-01 00:00:00,client_random_topn_5
2,22458302736,2026-12-01 00:00:00,client_random_topn_11
3,35893516279,2026-12-01 00:00:00,client_random_topn_4
4,45779035324,2026-12-01 00:00:00,fix_0.53_1


### 2. Вытаскиваем уникальные категории

In [40]:
%%time
with conn:
    categories_df = conn.fetch(f"""
        select distinct
            title_id,
            title
        from {cb_charges_compact}
        where title_id is not null
        order by title
    """)

categories_df["suffix"] = categories_df["title_id"].apply(category_suffix)

duplicates = (
    categories_df
    .groupby("suffix")
    .size()
    .reset_index(name="cnt")
    .query("cnt > 1")
)

duplicates

CPU times: user 754 ms, sys: 114 ms, total: 869 ms
Wall time: 9.81 s


,suffix,cnt


### 3. Общие траты + wide-траты в категориях

In [41]:
%%time
def generate_category_spend_sql(categories_df):
    sql_parts = []

    for _, row in categories_df.iterrows():
        category_id = row["title_id"]
        suffix = row["suffix"]

        sql_parts.append(f"""
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date = add_months(b.forecast_month, -2)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_lag1m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date = add_months(b.forecast_month, -3)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_lag2m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date = add_months(b.forecast_month, -4)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_lag3m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date >= add_months(b.forecast_month, -4)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_3m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.charge, 0) else 0
            end) as spend_cat_{suffix}_6m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date >= add_months(b.forecast_month, -4)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.count_charge, 0) else 0
            end) as cnt_cat_{suffix}_3m,
        sum(case
                when t.title_id = '{category_id}'
                 and t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.count_charge, 0) else 0
            end) as cnt_cat_{suffix}_6m
        """)

    return ",\n".join(sql_parts)

CPU times: user 17 μs, sys: 0 ns, total: 17 μs
Wall time: 30 μs


In [42]:
%%time
category_spend_sql = generate_category_spend_sql(categories_df)

drop = f"""
drop table if exists {cb_category_spend_wide}
"""

create = f"""
create table {cb_category_spend_wide} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,

    -- общие траты клиента, lag_1m = последний закрытый месяц
    sum(case
            when t.month_date = add_months(b.forecast_month, -2)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_1m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -3)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_2m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -4)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_3m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -5)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_4m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -6)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_5m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -7)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_lag_6m,
    sum(case
            when t.month_date >= add_months(b.forecast_month, -4)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_3m,
    sum(case
            when t.month_date >= add_months(b.forecast_month, -7)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.charge, 0) else 0
        end) as total_spend_6m,
    sum(case
            when t.month_date = add_months(b.forecast_month, -2)
            then coalesce(t.count_charge, 0) else 0
        end) as total_cnt_txn_lag_1m,
    sum(case
            when t.month_date >= add_months(b.forecast_month, -4)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.count_charge, 0) else 0
        end) as total_cnt_txn_3m,
    sum(case
            when t.month_date >= add_months(b.forecast_month, -7)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.count_charge, 0) else 0
        end) as total_cnt_txn_6m,
    -- wide-фичи по категориям
    {category_spend_sql}
from {cb_base} b
left join {cb_charges_compact} t
    on b.cdi_id = t.cdi_id
   and t.month_date >= add_months(b.forecast_month, -7)
   and t.month_date < add_months(b.forecast_month, -1)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_category_spend_wide, preview_table=cb_category_spend_wide)

CPU times: user 2.96 s, sys: 8.17 s, total: 11.1 s
Wall time: 14min 31s


/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,cdi_id,forecast_month,total_spend_lag_1m,total_spend_lag_2m,total_spend_lag_3m,total_spend_lag_4m,total_spend_lag_5m,total_spend_lag_6m,total_spend_3m,total_spend_6m,...,spend_cat_1130_6m,cnt_cat_1130_3m,cnt_cat_1130_6m,spend_cat_1131_lag1m,spend_cat_1131_lag2m,spend_cat_1131_lag3m,spend_cat_1131_3m,spend_cat_1131_6m,cnt_cat_1131_3m,cnt_cat_1131_6m
0,940898349,2026-12-01 00:00:00,0.00,410.00,0.00,0.00,0.00,0.00,410.00,410.00,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,0
1,31134150913,2026-12-01 00:00:00,0.00,5722.95,41523.01,52625.57,8905.14,0.00,47245.96,108776.67,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,0
2,34676043003,2026-12-01 00:00:00,3801.17,0.00,0.00,0.00,0.00,0.00,3801.17,3801.17,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,0
3,26725382833,2026-12-01 00:00:00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,0
4,1743386315,2026-12-01 00:00:00,0.00,0.00,9433.22,15218.19,8474.48,46803.65,9433.22,79929.54,...,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0,0


### 4. Предложенные категории

In [43]:
%%time
def generate_offered_percent_sql(categories_df):
    sql_parts = []

    for _, row in categories_df.iterrows():
        category_id = row["title_id"]
        suffix = row["suffix"]

        sql_parts.append(f"""
        max(case
                when o.category_id = '{category_id}'
                then cast(regexp_replace(cast(o.category_percent as string), ',', '.') as double)
                else 0
            end) as offered_percent_cat_{suffix}
        """)

    return ",\n".join(sql_parts)

offered_percent_sql = generate_offered_percent_sql(categories_df)

drop = f"""
drop table if exists {cb_offered_percent_wide}
"""

create = f"""
create table {cb_offered_percent_wide} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,
    b.offer_scenario_id,
    count(distinct o.category_id) as num_offered_categories,
    max(cast(regexp_replace(cast(o.category_percent as string), ',', '.') as double)) as max_offered_percent,
    avg(cast(regexp_replace(cast(o.category_percent as string), ',', '.') as double)) as avg_offered_percent,
    {offered_percent_sql}
from {cb_base_scen} b
left join {send_to_dbo} o
    on b.cdi_id = cast(o.cdi_id as bigint)
   and b.forecast_month = cast(o.period_start_date as timestamp)
   and b.offer_scenario_id = o.offer_scenario_id
group by
    b.cdi_id,
    b.forecast_month,
    b.offer_scenario_id
"""

run_sql(conn, [drop, create], compute_stats_table=cb_offered_percent_wide, preview_table=cb_offered_percent_wide)

CPU times: user 2.36 s, sys: 7.38 s, total: 9.73 s
Wall time: 13min 40s


/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,cdi_id,forecast_month,offer_scenario_id,num_offered_categories,max_offered_percent,avg_offered_percent,offered_percent_cat_1102,offered_percent_cat_1137,offered_percent_cat_1138,offered_percent_cat_1135,...,offered_percent_cat_1121,offered_percent_cat_1122,offered_percent_cat_1134,offered_percent_cat_1124,offered_percent_cat_1125,offered_percent_cat_1127,offered_percent_cat_1128,offered_percent_cat_1129,offered_percent_cat_1130,offered_percent_cat_1131
0,460620807,2026-12-01 00:00:00,fix_3.25_1,10,15.0,6.2,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0
1,1472985698,2026-12-01 00:00:00,client_random_topn_7,10,15.0,6.2,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,5.0,5.0,0.0,0.0,0.0,0.0,0.0
2,1207158639,2026-12-01 00:00:00,client_random_topn_10,10,15.0,6.2,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,0.0,5.0
3,1031304867,2026-12-01 00:00:00,fix_4.00_1,10,15.0,6.2,0.0,0.0,0.0,0.0,...,0.0,5.0,0.0,0.0,0.0,0.0,15.0,0.0,0.0,5.0
4,157901564,2026-12-01 00:00:00,fix_1.25_1,10,15.0,6.2,5.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0,5.0,0.0


### 5. История подключенных категорий

In [44]:
%%time
def generate_selected_history_sql(categories_df):
    parts = []

    for _, row in categories_df.iterrows():
        category_id = row["title_id"]
        suffix = row["suffix"]

        parts.append(f"""
        max(case
                when s.category_id = '{category_id}'
                 and trunc(cast(s.end_date as timestamp), 'MM') = add_months(b.forecast_month, -2)
                then 1 else 0
            end) as was_selected_cat_{suffix}_lag1m,
        max(case
                when s.category_id = '{category_id}'
                 and trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -4)
                 and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
                then 1 else 0
            end) as was_selected_cat_{suffix}_3m,
        max(case
                when s.category_id = '{category_id}'
                 and trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
                 and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
                then 1 else 0
            end) as was_selected_cat_{suffix}_6m,
        avg(case
                when s.category_id = '{category_id}'
                 and trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
                 and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
                then cast(regexp_replace(cast(s.category_percent as string), ',', '.') as double)
            end) as avg_selected_percent_cat_{suffix}_6m
        """)

    return ",\n".join(parts)

CPU times: user 2 μs, sys: 5 μs, total: 7 μs
Wall time: 12.2 μs


In [45]:
%%time
selected_history_sql = generate_selected_history_sql(categories_df)

drop = f"""
drop table if exists {cb_selected_history_wide}
"""

create = f"""
create table {cb_selected_history_wide} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,

    case
        when count(distinct case
            when trunc(cast(s.end_date as timestamp), 'MM') = add_months(b.forecast_month, -2)
            then s.category_id
        end) > 0
        then 1 else 0
    end as selected_any_lag_1m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') = add_months(b.forecast_month, -2)
        then s.category_id
    end) as cnt_selected_categories_lag_1m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -4)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then s.category_id
    end) as cnt_unique_selected_categories_3m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then s.category_id
    end) as cnt_unique_selected_categories_6m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -4)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then concat(cast(trunc(cast(s.end_date as timestamp), 'MM') as string), '', cast(s.category_id as string))
    end) as cnt_selected_category_months_3m,
    count(distinct case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then concat(cast(trunc(cast(s.end_date as timestamp), 'MM') as string), '', cast(s.category_id as string))
    end) as cnt_selected_category_months_6m,
    max(case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then cast(regexp_replace(cast(s.category_percent as string), ',', '.') as double)
    end) as max_selected_percent_6m,
    avg(case
        when trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
         and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
        then cast(regexp_replace(cast(s.category_percent as string), ',', '.') as double)
    end) as avg_selected_percent_6m,
    {selected_history_sql}
from {cb_base} b
left join {LOYALTY_CLIENT_CATEGORIES} s
    on b.cdi_id = cast(s.party_id as bigint)
   and trunc(cast(s.end_date as timestamp), 'MM') >= add_months(b.forecast_month, -7)
   and trunc(cast(s.end_date as timestamp), 'MM') < add_months(b.forecast_month, -1)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(
    conn,
    [drop, create],
    compute_stats_table=cb_selected_history_wide,
    preview_table=cb_selected_history_wide
)

CPU times: user 392 ms, sys: 602 ms, total: 994 ms
Wall time: 1min 4s


/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,cdi_id,forecast_month,selected_any_lag_1m,cnt_selected_categories_lag_1m,cnt_unique_selected_categories_3m,cnt_unique_selected_categories_6m,cnt_selected_category_months_3m,cnt_selected_category_months_6m,max_selected_percent_6m,avg_selected_percent_6m,...,was_selected_cat_1129_6m,avg_selected_percent_cat_1129_6m,was_selected_cat_1130_lag1m,was_selected_cat_1130_3m,was_selected_cat_1130_6m,avg_selected_percent_cat_1130_6m,was_selected_cat_1131_lag1m,was_selected_cat_1131_3m,was_selected_cat_1131_6m,avg_selected_percent_cat_1131_6m
0,44798368763,2026-12-01 00:00:00,0,0,0,0,0,0,NaN,NaN,...,0,None,0,0,0,None,0,0,0,None
1,1207500533,2026-12-01 00:00:00,0,0,0,0,0,0,NaN,NaN,...,0,None,0,0,0,None,0,0,0,None
2,1472487359,2026-12-01 00:00:00,0,0,0,0,0,0,NaN,NaN,...,0,None,0,0,0,None,0,0,0,None
3,46193008771,2026-12-01 00:00:00,0,0,5,5,5,5,5.0,4.2,...,0,None,0,0,0,None,0,0,0,None
4,44938554301,2026-12-01 00:00:00,0,0,0,0,0,0,NaN,NaN,...,0,None,0,0,0,None,0,0,0,None


### 6. ACRM targets

In [48]:
%%time

drop = f"""
drop table if exists {cb_acrm_monthly}
"""

create = f"""
create table {cb_acrm_monthly} stored as parquet as
select
    cast(a.cdi_id as bigint) as cdi_id,
    cast(trunc(cast(a.transaction_date as date), 'MM') as timestamp) as month_date,
    least(sum(coalesce(a.coins, 0)), 5000) as cashback_month,
    sum(coalesce(a.transaction_amount, 0)) as pos_turnover_month,
    count(*) as pos_txn_cnt_month
from sandbox_crft.pia_offline_coins_trunc_curr a
join (
    select distinct cdi_id
    from {cb_base}
) base
    on cast(a.cdi_id as bigint) = base.cdi_id
where cast(a.transaction_date as date) >= add_months(cast('{start_month}' as timestamp), -7)
  and cast(a.transaction_date as date) < add_months(cast('{end_month}' as timestamp), 1)
  and a.flg_rub_trans = 1
group by
    cast(a.cdi_id as bigint),
    cast(trunc(cast(a.transaction_date as date), 'MM') as timestamp)
"""

run_sql(
    conn,
    [drop, create],
    compute_stats_table=cb_acrm_monthly,
    preview_table=cb_acrm_monthly
)

CPU times: user 96.8 ms, sys: 40.5 ms, total: 137 ms
Wall time: 3.56 s


/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,cdi_id,month_date,cashback_month,pos_turnover_month,pos_txn_cnt_month
0,30323873798,2026-09-01 00:00:00,188,11937.94,49
1,1120587270,2026-07-01 00:00:00,287,14918.60,12
2,32179471269,2026-01-01 00:00:00,21,2113.99,1
3,27985495740,2026-01-01 00:00:00,35,4026.69,21
4,16439198576,2025-09-01 00:00:00,84,10223.00,51


In [49]:
%%time
drop = f"""
drop table if exists {cb_acrm_targets}
"""

create = f"""
create table {cb_acrm_targets} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,
    coalesce(t.cashback_month, 0) as target_cashback_month,
    coalesce(t.pos_turnover_month, 0) as target_pos_turnover_month,
    coalesce(t.pos_txn_cnt_month, 0) as target_pos_txn_cnt_month
from {cb_base} b
left join {cb_acrm_monthly} t
    on b.cdi_id = t.cdi_id
   and b.forecast_month = t.month_date
"""

run_sql(conn, [drop, create], compute_stats_table=cb_acrm_targets, preview_table=cb_acrm_targets)

CPU times: user 53 ms, sys: 27.3 ms, total: 80.3 ms
Wall time: 2.09 s


/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,cdi_id,forecast_month,target_cashback_month,target_pos_turnover_month,target_pos_txn_cnt_month
0,562061312,2026-12-01 00:00:00,0,0.0,0
1,2251296738,2026-12-01 00:00:00,0,0.0,0
2,1651960355,2026-12-01 00:00:00,0,0.0,0
3,45875857509,2026-12-01 00:00:00,0,0.0,0
4,158308509,2026-12-01 00:00:00,0,0.0,0


### 7. ACRM lags

In [50]:
%%time
drop = f"""
drop table if exists {cb_acrm_lags}
"""

create = f"""
create table {cb_acrm_lags} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,

    sum(case when t.month_date = add_months(b.forecast_month, -2)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_lag_1m,
    sum(case when t.month_date = add_months(b.forecast_month, -3)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_lag_2m,
    sum(case when t.month_date = add_months(b.forecast_month, -4)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_lag_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -7)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.cashback_month, 0) else 0 end) as cashback_6m,
    avg(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.cashback_month, 0) end) as cashback_avg_3m,
    stddev(case when t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.cashback_month, 0) end) as cashback_std_6m,
    sum(case when t.month_date = add_months(b.forecast_month, -2)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_lag_1m,
    sum(case when t.month_date = add_months(b.forecast_month, -3)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_lag_2m,
    sum(case when t.month_date = add_months(b.forecast_month, -4)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_lag_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -7)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_turnover_month, 0) else 0 end) as pos_turnover_6m,
    avg(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_turnover_month, 0) end) as pos_turnover_avg_3m,
    stddev(case when t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.pos_turnover_month, 0) end) as pos_turnover_std_6m,
    sum(case when t.month_date >= add_months(b.forecast_month, -4)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_txn_cnt_month, 0) else 0 end) as pos_txn_cnt_3m,
    sum(case when t.month_date >= add_months(b.forecast_month, -7)
              and t.month_date < add_months(b.forecast_month, -1)
             then coalesce(t.pos_txn_cnt_month, 0) else 0 end) as pos_txn_cnt_6m
from {cb_base} b
left join {cb_acrm_monthly} t
    on b.cdi_id = t.cdi_id
   and t.month_date >= add_months(b.forecast_month, -7)
   and t.month_date < add_months(b.forecast_month, -1)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_acrm_lags, preview_table=cb_acrm_lags)

CPU times: user 80.1 ms, sys: 51.1 ms, total: 131 ms
Wall time: 4.99 s


/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,cdi_id,forecast_month,cashback_lag_1m,cashback_lag_2m,cashback_lag_3m,cashback_3m,cashback_6m,cashback_avg_3m,cashback_std_6m,pos_turnover_lag_1m,pos_turnover_lag_2m,pos_turnover_lag_3m,pos_turnover_3m,pos_turnover_6m,pos_turnover_avg_3m,pos_turnover_std_6m,pos_txn_cnt_3m,pos_txn_cnt_6m
0,30973415649,2026-12-01 00:00:00,0,0,0,0,0,None,None,0.0,0.0,0.0,0.0,0.0,None,None,0,0
1,1561288345,2026-12-01 00:00:00,0,0,0,0,0,None,None,0.0,0.0,0.0,0.0,0.0,None,None,0,0
2,20551737582,2026-12-01 00:00:00,0,0,0,0,0,None,None,0.0,0.0,0.0,0.0,0.0,None,None,0,0
3,46268809803,2026-12-01 00:00:00,0,0,0,0,0,None,None,0.0,0.0,0.0,0.0,0.0,None,None,0,0
4,1743463522,2026-12-01 00:00:00,0,0,0,0,0,None,None,0.0,0.0,0.0,0.0,0.0,None,None,0,0


### 8. Activity feats

In [51]:
%%time
drop = f"""
drop table if exists {cb_activity_features}
"""

create = f"""
create table {cb_activity_features} stored as parquet as
select
    b.cdi_id,
    b.forecast_month,

    count(distinct case
        when t.month_date = add_months(b.forecast_month, -2)
         and coalesce(t.charge, 0) > 0
        then t.title_id
    end) as cnt_categories_with_spend_1m,
    count(distinct case
        when t.month_date >= add_months(b.forecast_month, -4)
         and t.month_date < add_months(b.forecast_month, -1)
         and coalesce(t.charge, 0) > 0
        then t.title_id
    end) as cnt_categories_with_spend_3m,
    count(distinct case
        when t.month_date >= add_months(b.forecast_month, -7)
         and t.month_date < add_months(b.forecast_month, -1)
         and coalesce(t.charge, 0) > 0
        then t.title_id
    end) as cnt_categories_with_spend_6m,
    stddev(case
        when t.month_date >= add_months(b.forecast_month, -7)
         and t.month_date < add_months(b.forecast_month, -1)
        then coalesce(t.charge, 0)
    end) as category_spend_std_6m,
    avg(case
        when t.month_date >= add_months(b.forecast_month, -7)
         and t.month_date < add_months(b.forecast_month, -1)
        then coalesce(t.charge, 0)
    end) as category_spend_avg_6m,
    case
        when avg(case
            when t.month_date >= add_months(b.forecast_month, -7)
             and t.month_date < add_months(b.forecast_month, -1)
            then coalesce(t.charge, 0)
        end) > 0
        then
            stddev(case
                when t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.charge, 0)
            end)
            /
            avg(case
                when t.month_date >= add_months(b.forecast_month, -7)
                 and t.month_date < add_months(b.forecast_month, -1)
                then coalesce(t.charge, 0)
            end)
        else 0
    end as category_spend_cv_6m
from {cb_base} b
left join {cb_charges_compact} t
    on b.cdi_id = t.cdi_id
   and t.month_date >= add_months(b.forecast_month, -7)
   and t.month_date < add_months(b.forecast_month, -1)
group by
    b.cdi_id,
    b.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_activity_features, preview_table=cb_activity_features)

CPU times: user 226 ms, sys: 439 ms, total: 666 ms
Wall time: 47.2 s


/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,cdi_id,forecast_month,cnt_categories_with_spend_1m,cnt_categories_with_spend_3m,cnt_categories_with_spend_6m,category_spend_std_6m,category_spend_avg_6m,category_spend_cv_6m
0,1207500533,2026-12-01 00:00:00,1,1,1,11.022704,0.694365,15.874509
1,42518744577,2026-12-01 00:00:00,0,0,0,NaN,NaN,0.000000
2,27531952210,2026-12-01 00:00:00,0,0,0,NaN,NaN,0.000000
3,44967459383,2026-12-01 00:00:00,2,5,6,540.453758,73.484921,7.354621
4,1561502774,2026-12-01 00:00:00,2,6,9,1076.752444,195.117540,5.518481


### 9. Потенциальный кешбэк

In [52]:
%%time
drop = f"""
drop table if exists {cb_potential_features}
"""

create = f"""
create table {cb_potential_features} stored as parquet as
with spend_3m_by_category as (
    select
        b.cdi_id,
        b.forecast_month,
        t.title_id as category_id,
        sum(coalesce(t.charge, 0)) as spend_3m,
        sum(coalesce(t.count_charge, 0)) as cnt_txn_3m
    from {cb_base} b
    join {cb_charges_compact} t
        on b.cdi_id = t.cdi_id
       and t.month_date >= add_months(b.forecast_month, -4)
       and t.month_date < add_months(b.forecast_month, -1)
    group by
        b.cdi_id,
        b.forecast_month,
        t.title_id
),
offered as (
    select
        cast(cdi_id as bigint) as cdi_id,
        cast(period_start_date as timestamp) as forecast_month,
        offer_scenario_id,
        category_id,
        max(cast(regexp_replace(cast(category_percent as string), ',', '.') as double)) as offered_percent
    from {send_to_dbo}
    group by
        cast(cdi_id as bigint),
        cast(period_start_date as timestamp),
        offer_scenario_id,
        category_id
)
select
    b.cdi_id,
    b.forecast_month,
    b.offer_scenario_id,
    sum(
        coalesce(s.spend_3m, 0) / 3.0
        * coalesce(o.offered_percent, 0) / 100.0
    ) as potential_cashback_sum_offered,
    max(
        coalesce(s.spend_3m, 0) / 3.0
        * coalesce(o.offered_percent, 0) / 100.0
    ) as potential_cashback_max_offered,
    avg(
        coalesce(s.spend_3m, 0) / 3.0
        * coalesce(o.offered_percent, 0) / 100.0
    ) as potential_cashback_avg_offered,
    count(distinct case
        when coalesce(s.spend_3m, 0) > 0
        then o.category_id
    end) as cnt_offered_categories_with_spend_3m,
    sum(coalesce(s.spend_3m, 0)) as offered_categories_spend_3m,
    sum(coalesce(s.cnt_txn_3m, 0)) as offered_categories_cnt_txn_3m
from {cb_base_scen} b
left join offered o
    on b.cdi_id = o.cdi_id
   and b.forecast_month = o.forecast_month
   and b.offer_scenario_id = o.offer_scenario_id
left join spend_3m_by_category s
    on o.cdi_id = s.cdi_id
   and o.forecast_month = s.forecast_month
   and o.category_id = s.category_id
group by
    b.cdi_id,
    b.forecast_month,
    b.offer_scenario_id
"""

run_sql(conn, [drop, create], compute_stats_table=cb_potential_features, preview_table=cb_potential_features)

CPU times: user 878 ms, sys: 2.68 s, total: 3.55 s
Wall time: 4min 43s


/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,cdi_id,forecast_month,offer_scenario_id,potential_cashback_sum_offered,potential_cashback_max_offered,potential_cashback_avg_offered,cnt_offered_categories_with_spend_3m,offered_categories_spend_3m,offered_categories_cnt_txn_3m
0,18533527604,2026-12-01 00:00:00,fix_0.53_1,0.0000,0.0000,0.00000,0,0.00,0
1,47254949718,2026-12-01 00:00:00,client_random_topn_13,0.0000,0.0000,0.00000,0,0.00,0
2,158232946,2026-12-01 00:00:00,client_random_topn_10,0.0000,0.0000,0.00000,0,0.00,0
3,45216434381,2026-12-01 00:00:00,fix_1.25_1,111.4905,111.4905,11.14905,1,2229.81,2
4,45058461710,2026-12-01 00:00:00,fix_0.75_1,0.0000,0.0000,0.00000,0,0.00,0


### 10. Share-фичи

In [53]:
%%time
categories_df

CPU times: user 1e+03 ns, sys: 8 μs, total: 9 μs
Wall time: 17.6 μs


,title_id,title,suffix
0,9a038f75-65c1-43b5-95bc-62967a8d1102,АЗС,1102
1,9a038f75-65c1-43b5-95bc-62967a8d1137,Автозапчасти,1137
2,9a038f75-65c1-43b5-95bc-62967a8d1138,Автоуслуги,1138
3,9a038f75-65c1-43b5-95bc-62967a8d1135,Аксессуары,1135
4,9a038f75-65c1-43b5-95bc-62967a8d1141,Активный отдых,1141
5,9a038f75-65c1-43b5-95bc-62967a8d1103,Алкоголь,1103
6,9a038f75-65c1-43b5-95bc-62967a8d1104,Аптеки,1104
7,9a038f75-65c1-43b5-95bc-62967a8d1139,Аренда авто,1139
8,9a038f75-65c1-43b5-95bc-62967a8d1105,Бытовая техника и электроника,1105
9,9a038f75-65c1-43b5-95bc-62967a8d1140,Бытовые услуги,1140


In [54]:
%%time
def generate_share_sql(categories_df):
    parts = []

    for _, row in categories_df.iterrows():
        suffix = row["suffix"]

        parts.append(f"""
        case
            when coalesce(total_spend_3m, 0) > 0
            then coalesce(spend_cat_{suffix}_3m, 0) / total_spend_3m
            else 0
        end as share_spend_cat_{suffix}_3m
        """)

    return ",\n".join(parts)


share_sql = generate_share_sql(categories_df)

CPU times: user 2.15 ms, sys: 0 ns, total: 2.15 ms
Wall time: 2.17 ms


In [55]:
%%time
drop = f"""
drop table if exists {cb_share_features}
"""

create = f"""
create table {cb_share_features} stored as parquet as
select
    cdi_id,
    forecast_month,
    {share_sql}
from {cb_category_spend_wide}
"""

run_sql(conn, [drop, create], compute_stats_table=cb_share_features, preview_table=cb_share_features)

CPU times: user 128 ms, sys: 66.2 ms, total: 194 ms
Wall time: 6.31 s


/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,cdi_id,forecast_month,share_spend_cat_1102_3m,share_spend_cat_1137_3m,share_spend_cat_1138_3m,share_spend_cat_1135_3m,share_spend_cat_1141_3m,share_spend_cat_1103_3m,share_spend_cat_1104_3m,share_spend_cat_1139_3m,...,share_spend_cat_1121_3m,share_spend_cat_1122_3m,share_spend_cat_1134_3m,share_spend_cat_1124_3m,share_spend_cat_1125_3m,share_spend_cat_1127_3m,share_spend_cat_1128_3m,share_spend_cat_1129_3m,share_spend_cat_1130_3m,share_spend_cat_1131_3m
0,940898349,2026-12-01 00:00:00,0.00000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,1.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
1,31134150913,2026-12-01 00:00:00,0.00000,0.0,0.0,0.0,0.061669,0.0,0.000000,0.0,...,0.0,0.0,0.075242,0.0,0.0,0.0,0.068137,0.0,0.0,0.0
2,34676043003,2026-12-01 00:00:00,0.00000,0.0,0.0,0.0,0.000000,0.0,0.039462,0.0,...,0.0,0.0,0.827948,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
3,26725382833,2026-12-01 00:00:00,0.00000,0.0,0.0,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
4,1743386315,2026-12-01 00:00:00,0.13317,0.0,0.0,0.0,0.000000,0.0,0.468981,0.0,...,0.0,0.0,0.119471,0.0,0.0,0.0,0.083535,0.0,0.0,0.0


In [56]:
%%time
share_cols_sql = ",\n    ".join([
    f"coalesce(sh.share_spend_cat_{row['suffix']}_3m, 0) as sharespend_cat_{row['suffix']}_3m"
    for _, row in categories_df.iterrows()
])

selected_history_cols = []

for _, row in categories_df.iterrows():
    suffix = row["suffix"]

    selected_history_cols.extend([
        f"coalesce(sel.was_selected_cat_{suffix}_lag1m, 0) as was_selected_cat_{suffix}_lag1m",
        f"coalesce(sel.was_selected_cat_{suffix}_3m, 0) as wasselected_cat_{suffix}_3m",
        f"coalesce(sel.was_selected_cat_{suffix}_6m, 0) as wasselected_cat_{suffix}_6m",
        f"coalesce(sel.avg_selected_percent_cat_{suffix}_6m, 0) as avgselected_percent_cat_{suffix}_6m",
    ])

selected_history_cols_sql = ",\n    ".join(selected_history_cols)

CPU times: user 11.9 ms, sys: 407 μs, total: 12.3 ms
Wall time: 11.3 ms


In [57]:
%%time
spend_wide_cols = []

for _, row in categories_df.iterrows():
    suffix = row["suffix"]
    spend_wide_cols.extend([
        f"coalesce(sp.spend_cat_{suffix}_lag1m, 0) as spend_cat_{suffix}_lag1m",
        f"coalesce(sp.spend_cat_{suffix}_lag2m, 0) as spend_cat_{suffix}_lag2m",
        f"coalesce(sp.spend_cat_{suffix}_lag3m, 0) as spend_cat_{suffix}_lag3m",
        f"coalesce(sp.spend_cat_{suffix}_3m, 0) as spendcat_{suffix}_3m",
        f"coalesce(sp.spend_cat_{suffix}_6m, 0) as spendcat_{suffix}_6m",
        f"coalesce(sp.cnt_cat_{suffix}_3m, 0) as cntcat_{suffix}_3m",
        f"coalesce(sp.cnt_cat_{suffix}_6m, 0) as cntcat_{suffix}_6m",
    ])

spend_wide_cols_sql = ",\n    ".join(spend_wide_cols)


offered_wide_cols = []

for _, row in categories_df.iterrows():
    suffix = row["suffix"]
    offered_wide_cols.append(
        f"coalesce(off.offered_percent_cat_{suffix}, 0) as offered_percent_cat_{suffix}"
    )

offered_wide_cols_sql = ",\n    ".join(offered_wide_cols)

CPU times: user 11.5 ms, sys: 1.68 ms, total: 13.2 ms
Wall time: 11.5 ms


### 11. Финальная витрина

In [58]:
%%time
drop = f"""
drop table if exists {cb_dataset}
"""

create = f"""
create table {cb_dataset} stored as parquet as
with loyalty_connected as (
    select
        cast(party_id as bigint) as cdi_id,
        trunc(cast(end_date as timestamp), 'MM') as forecast_month
    from {LOYALTY_CLIENT_CATEGORIES}
    group by
        cast(party_id as bigint),
        trunc(cast(end_date as timestamp), 'MM')
)
select
    b.cdi_id,
    b.forecast_month,
    b.offer_scenario_id,
    -- targets
    coalesce(trg.target_cashback_month, 0) as target_cashback_month,
    coalesce(trg.target_pos_turnover_month, 0) as target_pos_turnover_month,
    coalesce(trg.target_pos_txn_cnt_month, 0) as target_pos_txn_cnt_month,
    case
        when lc.cdi_id is not null then 1
        else 0
    end as target_loyalty_connected_flg,
    -- общие траты клиента
    coalesce(sp.total_spend_lag_1m, 0) as total_spend_lag_1m,
    coalesce(sp.total_spend_lag_2m, 0) as total_spend_lag_2m,
    coalesce(sp.total_spend_lag_3m, 0) as total_spend_lag_3m,
    coalesce(sp.total_spend_lag_4m, 0) as total_spend_lag_4m,
    coalesce(sp.total_spend_lag_5m, 0) as total_spend_lag_5m,
    coalesce(sp.total_spend_lag_6m, 0) as total_spend_lag_6m,
    coalesce(sp.total_spend_3m, 0) as total_spend_3m,
    coalesce(sp.total_spend_6m, 0) as total_spend_6m,
    coalesce(sp.total_cnt_txn_lag_1m, 0) as total_cnt_txn_lag_1m,
    coalesce(sp.total_cnt_txn_3m, 0) as total_cnt_txn_3m,
    coalesce(sp.total_cnt_txn_6m, 0) as total_cnt_txn_6m,
    -- wide-траты по категориям
    {spend_wide_cols_sql},
    -- предложенные категории на forecast_month
    coalesce(off.num_offered_categories, 0) as num_offered_categories,
    coalesce(off.max_offered_percent, 0) as max_offered_percent,
    coalesce(off.avg_offered_percent, 0) as avg_offered_percent,
    {offered_wide_cols_sql},
    -- история выбранных категорий
    coalesce(sel.selected_any_lag_1m, 0) as selected_any_lag_1m,
    coalesce(sel.cnt_selected_categories_lag_1m, 0) as cnt_selected_categories_lag_1m,
    coalesce(sel.cnt_unique_selected_categories_3m, 0) as cnt_unique_selected_categories_3m,
    coalesce(sel.cnt_unique_selected_categories_6m, 0) as cnt_unique_selected_categories_6m,
    coalesce(sel.max_selected_percent_6m, 0) as max_selected_percent_6m,
    coalesce(sel.avg_selected_percent_6m, 0) as avg_selected_percent_6m,
    {selected_history_cols_sql},
    -- ACRM lags
    coalesce(al.cashback_lag_1m, 0) as cashback_lag_1m,
    coalesce(al.cashback_lag_2m, 0) as cashback_lag_2m,
    coalesce(al.cashback_lag_3m, 0) as cashback_lag_3m,
    coalesce(al.cashback_3m, 0) as cashback_3m,
    coalesce(al.cashback_6m, 0) as cashback_6m,
    coalesce(al.cashback_avg_3m, 0) as cashback_avg_3m,
    coalesce(al.cashback_std_6m, 0) as cashback_std_6m,
    coalesce(al.pos_turnover_lag_1m, 0) as pos_turnover_lag_1m,
    coalesce(al.pos_turnover_lag_2m, 0) as pos_turnover_lag_2m,
    coalesce(al.pos_turnover_lag_3m, 0) as pos_turnover_lag_3m,
    coalesce(al.pos_turnover_3m, 0) as pos_turnover_3m,
    coalesce(al.pos_turnover_6m, 0) as pos_turnover_6m,
    coalesce(al.pos_turnover_avg_3m, 0) as pos_turnover_avg_3m,
    coalesce(al.pos_turnover_std_6m, 0) as pos_turnover_std_6m,
    coalesce(al.pos_txn_cnt_3m, 0) as pos_txn_cnt_3m,
    coalesce(al.pos_txn_cnt_6m, 0) as pos_txn_cnt_6m,
    -- активность
    coalesce(act.cnt_categories_with_spend_1m, 0) as cnt_categories_with_spend_1m,
    coalesce(act.cnt_categories_with_spend_3m, 0) as cnt_categories_with_spend_3m,
    coalesce(act.cnt_categories_with_spend_6m, 0) as cnt_categories_with_spend_6m,
    coalesce(act.category_spend_std_6m, 0) as category_spend_std_6m,
    coalesce(act.category_spend_avg_6m, 0) as category_spend_avg_6m,
    coalesce(act.category_spend_cv_6m, 0) as category_spend_cv_6m,
    -- potential cashback
    coalesce(p.potential_cashback_sum_offered, 0) as potential_cashback_sum_offered,
    coalesce(p.potential_cashback_max_offered, 0) as potential_cashback_max_offered,
    coalesce(p.potential_cashback_avg_offered, 0) as potential_cashback_avg_offered,
    coalesce(p.cnt_offered_categories_with_spend_3m, 0) as cnt_offered_categories_with_spend_3m,
    coalesce(p.offered_categories_spend_3m, 0) as offered_categories_spend_3m,
    coalesce(p.offered_categories_cnt_txn_3m, 0) as offered_categories_cnt_txn_3m,
    -- share categories
    {share_cols_sql},
    -- calendar
    year(b.forecast_month) as forecast_year,
    month(b.forecast_month) as forecast_month_num
from {cb_base_scen} b
left join {cb_acrm_targets} trg
    on b.cdi_id = trg.cdi_id
   and b.forecast_month = trg.forecast_month
left join {cb_category_spend_wide} sp
    on b.cdi_id = sp.cdi_id
   and b.forecast_month = sp.forecast_month
left join {cb_offered_percent_wide} off
    on b.cdi_id = off.cdi_id
   and b.forecast_month = off.forecast_month
   and b.offer_scenario_id = off.offer_scenario_id
left join {cb_selected_history_wide} sel
    on b.cdi_id = sel.cdi_id
   and b.forecast_month = sel.forecast_month
left join {cb_acrm_lags} al
    on b.cdi_id = al.cdi_id
   and b.forecast_month = al.forecast_month
left join {cb_activity_features} act
    on b.cdi_id = act.cdi_id
   and b.forecast_month = act.forecast_month
left join {cb_potential_features} p
    on b.cdi_id = p.cdi_id
   and b.forecast_month = p.forecast_month
   and b.offer_scenario_id = p.offer_scenario_id
left join {cb_share_features} sh
    on b.cdi_id = sh.cdi_id
   and b.forecast_month = sh.forecast_month
left join loyalty_connected lc
    on b.cdi_id = lc.cdi_id
   and b.forecast_month = lc.forecast_month
"""

run_sql(conn, [drop, create], compute_stats_table=cb_dataset, preview_table=cb_dataset)

CPU times: user 2.44 s, sys: 6.33 s, total: 8.77 s
Wall time: 11min 21s


/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,cdi_id,forecast_month,offer_scenario_id,target_cashback_month,target_pos_turnover_month,target_pos_txn_cnt_month,target_loyalty_connected_flg,total_spend_lag_1m,total_spend_lag_2m,total_spend_lag_3m,...,sharespend_cat_1134_3m,sharespend_cat_1124_3m,sharespend_cat_1125_3m,sharespend_cat_1127_3m,sharespend_cat_1128_3m,sharespend_cat_1129_3m,sharespend_cat_1130_3m,sharespend_cat_1131_3m,forecast_year,forecast_month_num
0,30481400176,2026-12-01 00:00:00,fix_0.78_1,0,0.0,0,0,0.00,241.89,5188.27,...,0.050621,0.0,0.0,0.0,0.172923,0.0,0.0,0.0,2026,12
1,30481400176,2026-12-01 00:00:00,fix_2.33_1,0,0.0,0,0,0.00,241.89,5188.27,...,0.050621,0.0,0.0,0.0,0.172923,0.0,0.0,0.0,2026,12
2,1120085687,2026-12-01 00:00:00,fix_2.67_1,0,0.0,0,0,2403.05,4228.92,40571.65,...,0.454559,0.0,0.0,0.0,0.011016,0.0,0.0,0.0,2026,12
3,1120085687,2026-12-01 00:00:00,client_random_topn_12,0,0.0,0,0,2403.05,4228.92,40571.65,...,0.454559,0.0,0.0,0.0,0.011016,0.0,0.0,0.0,2026,12
4,1120085687,2026-12-01 00:00:00,client_random_topn_7,0,0.0,0,0,2403.05,4228.92,40571.65,...,0.454559,0.0,0.0,0.0,0.011016,0.0,0.0,0.0,2026,12


In [59]:
%%time
cb_dataset

CPU times: user 0 ns, sys: 6 μs, total: 6 μs
Wall time: 11.4 μs


'sandbox_crft.cb_client_month_dataset_dec_active_base_dynamic_2026_rub_filtered'

### 12. JOIN фичестора

In [60]:
%%time
with conn:
    d1 = conn.fetch(f'''select * from {cb_dataset} where forecast_month = '{TARGET_MONTH}' limit 10''')
    df_all = conn.fetch(f'''select count(*) from {cb_dataset} limit 5''')
d1

/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


CPU times: user 773 ms, sys: 13.5 ms, total: 787 ms
Wall time: 1.84 s


,cdi_id,forecast_month,offer_scenario_id,target_cashback_month,target_pos_turnover_month,target_pos_txn_cnt_month,target_loyalty_connected_flg,total_spend_lag_1m,total_spend_lag_2m,total_spend_lag_3m,...,sharespend_cat_1134_3m,sharespend_cat_1124_3m,sharespend_cat_1125_3m,sharespend_cat_1127_3m,sharespend_cat_1128_3m,sharespend_cat_1129_3m,sharespend_cat_1130_3m,sharespend_cat_1131_3m,forecast_year,forecast_month_num
0,36269122923,2026-12-01 00:00:00,fix_1.75_1,0,0.0,0,0,0.0,19913.47,20939.25,...,0.558838,0.000000,0.0,0.001762,0.028321,0.0,0.0,0.0,2026,12
1,36269122923,2026-12-01 00:00:00,fix_2.33_1,0,0.0,0,0,0.0,19913.47,20939.25,...,0.558838,0.000000,0.0,0.001762,0.028321,0.0,0.0,0.0,2026,12
2,36269122923,2026-12-01 00:00:00,fix_1.50_1,0,0.0,0,0,0.0,19913.47,20939.25,...,0.558838,0.000000,0.0,0.001762,0.028321,0.0,0.0,0.0,2026,12
3,36269122923,2026-12-01 00:00:00,fix_0.75_1,0,0.0,0,0,0.0,19913.47,20939.25,...,0.558838,0.000000,0.0,0.001762,0.028321,0.0,0.0,0.0,2026,12
4,36269122923,2026-12-01 00:00:00,fix_0.61_1,0,0.0,0,0,0.0,19913.47,20939.25,...,0.558838,0.000000,0.0,0.001762,0.028321,0.0,0.0,0.0,2026,12
5,36269122923,2026-12-01 00:00:00,fix_0.97_1,0,0.0,0,0,0.0,19913.47,20939.25,...,0.558838,0.000000,0.0,0.001762,0.028321,0.0,0.0,0.0,2026,12
6,45401430510,2026-12-01 00:00:00,client_random_topn_18,0,0.0,0,0,0.0,10433.74,0.00,...,0.055048,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,2026,12
7,1384736992,2026-12-01 00:00:00,fix_0.53_1,0,0.0,0,0,17561.0,399.00,2814.00,...,0.000000,0.250746,0.0,0.000000,0.000000,0.0,0.0,0.0,2026,12
8,1384736992,2026-12-01 00:00:00,client_random_topn_4,0,0.0,0,0,17561.0,399.00,2814.00,...,0.000000,0.250746,0.0,0.000000,0.000000,0.0,0.0,0.0,2026,12
9,36338649878,2026-12-01 00:00:00,client_random_topn_11,0,0.0,0,0,0.0,18418.85,0.00,...,0.410332,0.000000,0.0,0.005212,0.000000,0.0,0.0,0.0,2026,12


In [61]:
%%time
df_all

CPU times: user 1e+03 ns, sys: 5 μs, total: 6 μs
Wall time: 10.3 μs


,count(*)
0,38313825


In [62]:
MODULAR_FEATURE_STORE = "sandbox_crft.feature_store_inc_modular"
with conn:
    d3 = conn.fetch(f"SELECT * FROM {MODULAR_FEATURE_STORE} LIMIT 1")
d3

/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,cdi_id,transactions__module_cust_id,transactions__category_supermarkets_sum,transactions__category_clothing_sum,transactions__category_pharmacies_sum,transactions__category_gas_sum,transactions__category_auto_sum,transactions__category_home_sum,transactions__category_traveling_sum,transactions__category_fast_food_sum,...,loyalty__spend_share_cat_taxi_3m_percent,loyalty__spend_share_cat_cinema_3m_percent,loyalty__spend_share_cat_transport_3m_percent,loyalty__spend_share_cat_fastfood_3m_percent,loyalty__spend_share_cat_hobby_3m_percent,loyalty__spend_share_cat_flowers_3m_percent,loyalty__spend_share_cat_jewelry_3m_percent,loyalty__load_dttm,report_date,hash_num
0,19502347191,None,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,None,2024-06-01,35


In [63]:
# Отбираем признаки modular FS по правилам обучения
base_cols = list(d1.columns)

modular_cols = [
    c for c in d3.columns
    if c not in set(base_cols)
    and c not in {"hash_num", "cdi_id", "report_date"}
    and not c.startswith("loyalty__")
    and not c.endswith(("_date", "_dttm", "_cust_id"))
]

print("base:", len(base_cols), "modular:", len(modular_cols))

base: 603 modular: 744


In [ ]:
# Для modular FS используем явную квалификацию ld / cs, чтобы исключить неоднозначность имён.

In [64]:
select_clause = ",\n        ".join(
    [f"ld.`{name}` AS `{name}`" for name in base_cols] +
    [f"cs.`{name}` AS `{name}`" for name in modular_cols]
)

In [65]:
conn_hive = connect(to="IMPALA")  # текущая Kerberos-сессия; без пароля в ноутбуке

Successfully login!


In [66]:
%%time
DATASET_WITH_FEATURES = f'sandbox_crft.mae_nbc_dataset_with_features_{postfix}'

CPU times: user 4 μs, sys: 15 μs, total: 19 μs
Wall time: 35.8 μs


In [67]:
%%time

with conn:
    fs_check = conn.fetch(f"""
        SELECT
            COUNT(*) AS rows_cnt,
            COUNT(DISTINCT CAST(cdi_id AS BIGINT)) AS clients_cnt
        FROM {MODULAR_FEATURE_STORE}
        WHERE CAST(report_date AS DATE) >= '2026-10-01'
          AND CAST(report_date AS DATE) < '2026-11-01'
    """)

fs_check

CPU times: user 77.6 ms, sys: 32.9 ms, total: 110 ms
Wall time: 4.12 s


,rows_cnt,clients_cnt
0,34324583,34324583


In [68]:
%%time
with conn:
    conn.execute(f'''drop table if exists {DATASET_WITH_FEATURES}''')

CPU times: user 26.5 ms, sys: 6.11 ms, total: 32.6 ms
Wall time: 163 ms


In [69]:
with conn_hive:
    conn_hive.execute(f'''CREATE TABLE {DATASET_WITH_FEATURES}
        STORED AS PARQUET TBLPROPERTIES('transactional'='false') AS
        SELECT {select_clause}
        FROM {cb_dataset} ld
        LEFT JOIN {MODULAR_FEATURE_STORE} cs
          ON CAST(cs.cdi_id AS BIGINT) = CAST(ld.cdi_id AS BIGINT)
         AND YEAR(CAST(cs.report_date AS DATE))*100 + MONTH(CAST(cs.report_date AS DATE))
             = YEAR(ADD_MONTHS(CAST(ld.forecast_month AS DATE),-2))*100
               + MONTH(ADD_MONTHS(CAST(ld.forecast_month AS DATE),-2))''')

In [70]:
%%time

with conn:
    check = conn.fetch(f"""
        SELECT COUNT(*) AS rows_cnt
        FROM {DATASET_WITH_FEATURES}
    """)

check

CPU times: user 83.9 ms, sys: 8.55 ms, total: 92.5 ms
Wall time: 1.12 s


,rows_cnt
0,38313825


In [71]:
# %%time
# MUST_HAVE = get_feature_list('data_fm_must_have_features')
# COLS_BIN_LOYAL = get_feature_list('train_pipeline_loyalty_binary_columns')
# COLS_CASHBACK = get_feature_list('train_pipeline_cashback_columns')
# SELECT_POS_LOYAL_COLS = get_feature_list('data_fm_pos_loyalty_columns')
# SELECT_POS_NOLOYAL_COLS = get_feature_list('data_fm_pos_noloyalty_columns')
# SELECT_DELTA_LOYAL_COLS = get_feature_list('data_fm_delta_loyalty_columns')
# SELECT_DELTA_NOLOYAL_COLS = get_feature_list('data_fm_delta_noloyalty_columns')
# SELECT_BINARY_LOYAL_COLS = get_feature_list('data_fm_binary_loyalty_columns')
# SELECT_BINARY_NOLOYAL_COLS = get_feature_list('data_fm_binary_noloyalty_columns')

In [72]:
total_cols_sql = "*"
# Dynamic: полный набор колонок сохраняем в Parquet; модельные колонки читает PyArrow по feature_names_.

In [73]:
%%time
cb_infer_data = f"sandbox_crft.cb_infer_dataset_{postfix}"

drop = f"""
    drop table if exists {cb_infer_data}
"""

create = f"""
    create table {cb_infer_data} stored as parquet as
    select
        {total_cols_sql}
    from {DATASET_WITH_FEATURES} d
"""

with conn_hive:
    for query in [drop, create]:
        conn_hive.execute(query)

CPU times: user 4.57 s, sys: 14 s, total: 18.5 s
Wall time: 26min 3s


In [74]:
import subprocess
import shutil

INFER_TABLE_NAME = cb_infer_data.split(".")[-1]

HDFS_INFER_DATASET = (
    f"/warehouse/tablespace/managed/hive/"
    f"sandbox_crft.db/{INFER_TABLE_NAME}"
)

result = subprocess.run(
    ["hdfs", "dfs", "-du", "-s", "-h", HDFS_INFER_DATASET],
    capture_output=True,
    text=True,
    check=True
)

print("Размер таблицы в HDFS:")
print(result.stdout)

disk = shutil.disk_usage(PROJECT_ROOT)

print(
    "Свободно на диске JupyterHub:",
    round(disk.free / 1024**3, 2),
    "ГБ"
)

Размер таблицы в HDFS:
1.4 G  4.3 G  /warehouse/tablespace/managed/hive/sandbox_crft.db/cb_infer_dataset_dec_active_base_dynamic_2026_rub_filtered

Свободно на диске JupyterHub: 329.66 ГБ


In [75]:
# Копируем строго созданную выше витрину. Никаких фиксированных июльских/ноябрьских имён.
import shutil, subprocess
LOCAL_INFER_PARENT = PROJECT_ROOT / "data" / "datasets" / "inference"
LOCAL_INFER_PARENT.mkdir(parents=True, exist_ok=True)
INFER_TABLE_NAME = cb_infer_data.split(".")[-1]
LOCAL_INFER_DATASET = LOCAL_INFER_PARENT / INFER_TABLE_NAME
HDFS_INFER_DATASET = f"/warehouse/tablespace/managed/hive/sandbox_crft.db/{INFER_TABLE_NAME}"
if LOCAL_INFER_DATASET.exists():
    shutil.rmtree(LOCAL_INFER_DATASET)
subprocess.run(["hdfs", "dfs", "-copyToLocal", HDFS_INFER_DATASET, str(LOCAL_INFER_PARENT)], check=True)
if not LOCAL_INFER_DATASET.is_dir():
    raise FileNotFoundError(LOCAL_INFER_DATASET)
print("Parquet:", LOCAL_INFER_DATASET)


Parquet: /home/jovyan/work/loyalty_handoff/dynamic/data/datasets/inference/cb_infer_dataset_dec_active_base_dynamic_2026_rub_filtered


In [ ]:
%%time
cb_infer_data

In [ ]:
%%time
!mkdir -p ./data/datasets/inference

In [ ]:
# Скачивание Parquet выполняется единым блоком ниже, по имени только что созданной таблицы.

In [ ]:
# Скачивание Parquet выполняется единым блоком ниже, по имени только что созданной таблицы.

In [ ]:
1 

## Скоринг

In [76]:
%%time
# Настройки inference вынесены в configs/inference_config.json
from nbc_project.inference_helpers import (
    load_inference_config,
    build_inference_table_names,
    run_sql,
    category_suffix,
    add_category_suffix,
    generate_category_spend_sql,
    generate_offered_percent_sql,
    generate_selected_history_sql,
    generate_share_sql,
    generate_send_to_dbo_scenario_mart_sql,
    add_offer_scenario_filter,
    read_partitioned_parquet_folder,
    optimize_dataframe_types,
    prepare_inference_frame,
    load_catboost_models,
    load_optional_calibrators,
    model_output_columns,
    predict_with_models,
    infer_required_input_columns,
    score_inference_parquet_in_batches,
    upload_folder_to_hdfs,
)

INFERENCE_CONFIG = load_inference_config(PROJECT_ROOT)
INFERENCE_CONFIG["preprocessing_mode"] = "training_aligned"
INFERENCE_CONFIG["strict_features"] = True


CPU times: user 21.9 ms, sys: 2.35 ms, total: 24.3 ms
Wall time: 66.8 ms


In [79]:
from pathlib import Path

# Модели из текущего обучающего запуска
MODEL_DIR = RUN_DIR / "models"

model_files = {
    "pos_loyal": "pos/model_T_202610_dynamic_psi_importance_auto_n.cbm",
    "pos_noloyal": "pos/model_C_202610_dynamic_psi_importance_auto_n.cbm",

    "binary_loyal": "binary_bin/model_T_202610_dynamic_psi_importance_auto_n.cbm",
    "binary_noloyal": "binary_bin/model_C_202610_dynamic_psi_importance_auto_n.cbm",

    "delta_loyal": "delta_bin/model_T_202610_dynamic_psi_importance_auto_n.cbm",
    "delta_noloyal": "delta_bin/model_C_202610_dynamic_psi_importance_auto_n.cbm",

    "cashback_connected": "train_pipeline/cashback_connected.cbm",
    "loyalty_connection": "train_pipeline/loyalty_connect.cbm",
}

assert set(model_files) == set(INFERENCE_CONFIG["model_paths"])

for model_name, filename in model_files.items():
    model_path = MODEL_DIR / filename

    assert model_path.is_file(), f"Не найдена модель: {model_path}"

    INFERENCE_CONFIG["model_paths"][model_name]["path"] = (
        model_path.relative_to(PROJECT_ROOT).as_posix()
    )

# Калибратор вероятности подключения лояльности
calibrator_path = (
    MODEL_DIR / "train_pipeline" / "isotonic_calibrator_loyalty.pkl"
)

assert calibrator_path.is_file(), f"Не найден калибратор: {calibrator_path}"

INFERENCE_CONFIG["model_paths"]["loyalty_connection"]["calibrator_path"] = (
    calibrator_path.relative_to(PROJECT_ROOT).as_posix()
)

# Проверяем результат
for model_name, spec in INFERENCE_CONFIG["model_paths"].items():
    print(f"{model_name}: {spec['path']}")

    if spec.get("calibrator_path"):
        print(f"  calibrator: {spec['calibrator_path']}")

print("\nВсе 8 моделей и калибратор найдены.")

pos_loyal: experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/pos/model_T_202610_dynamic_psi_importance_auto_n.cbm
pos_noloyal: experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/pos/model_C_202610_dynamic_psi_importance_auto_n.cbm
binary_loyal: experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/binary_bin/model_T_202610_dynamic_psi_importance_auto_n.cbm
binary_noloyal: experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/binary_bin/model_C_202610_dynamic_psi_importance_auto_n.cbm
delta_loyal: experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/delta_bin/model_T_202610_dynamic_psi_importance_auto_n.cbm
delta_noloyal: experiment_outputs

In [77]:
for model_name, spec in INFERENCE_CONFIG["model_paths"].items():
    print(f"{model_name}: {spec['path']}")
    if spec.get("calibrator_path"):
        print(f"  calibrator: {spec['calibrator_path']}")

pos_loyal: experiment_outputs/202610_dynamic/models/pos/model_T_202610_dynamic_psi_rfe_loss_auto_n.cbm
pos_noloyal: experiment_outputs/202610_dynamic/models/pos/model_C_202610_dynamic_psi_rfe_loss_auto_n.cbm
binary_loyal: experiment_outputs/202610_dynamic/models/binary_bin/model_T_202610_dynamic_psi_importance_auto_n.cbm
binary_noloyal: experiment_outputs/202610_dynamic/models/binary_bin/model_C_202610_dynamic_psi_importance_auto_n.cbm
delta_loyal: experiment_outputs/202610_dynamic/models/delta_bin/model_T_202610_dynamic_psi_importance_auto_n.cbm
delta_noloyal: experiment_outputs/202610_dynamic/models/delta_bin/model_C_202610_dynamic_psi_importance_auto_n.cbm
cashback_connected: experiment_outputs/202610_dynamic/models/train_pipeline/cashback_connected.cbm
loyalty_connection: experiment_outputs/202610_dynamic/models/train_pipeline/loyalty_connect.cbm
  calibrator: experiment_outputs/202610_dynamic/models/train_pipeline/isotonic_calibrator_loyalty.pkl


In [87]:
model_cat_features = {
    model.feature_names_[i]
    for model in models.values()
    for i in model.get_cat_feature_indices()
}

INFERENCE_CONFIG["cat_features"] = sorted(
    set(INFERENCE_CONFIG["cat_features"]) | model_cat_features
)

print("Категориальных признаков моделей:", len(model_cat_features))
print("Признаков в конфигурации:", len(INFERENCE_CONFIG["cat_features"]))

Категориальных признаков моделей: 9
Признаков в конфигурации: 16


In [78]:
from pathlib import Path

print("RUN_DIR:", RUN_DIR)

for model_group in [
    "pos",
    "binary_bin",
    "delta_bin",
    "train_pipeline",
]:
    model_dir = RUN_DIR / "models" / model_group

    files = sorted(
        list(model_dir.glob("*.cbm"))
        + list(model_dir.glob("*.pkl"))
    )

    print(f"\n{model_group}: {len(files)} файлов")

    for file in files:
        print(" ", file.relative_to(PROJECT_ROOT))

RUN_DIR: /home/jovyan/work/loyalty_handoff/dynamic/experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880

pos: 4 файлов
  experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/pos/model_C_202610_dynamic_psi_importance_auto_n.cbm
  experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/pos/model_C_202610_dynamic_psi_importance_auto_n.pkl
  experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/pos/model_T_202610_dynamic_psi_importance_auto_n.cbm
  experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__test_none__20261122_075913_858880/models/pos/model_T_202610_dynamic_psi_importance_auto_n.pkl

binary_bin: 4 файлов
  experiment_outputs/202610_dynamic_rub_filtered/train_202509_202609__valid_202610__

In [93]:
%%time
# Режимы inference. Для больших данных RUN_BATCH_INFERENCE=True.
RUN_BATCH_INFERENCE = True
RUN_FULL_INFERENCE = False

INFERENCE_DATASET_PATH = str(LOCAL_INFER_DATASET)
BATCH_INFERENCE_OUTPUT_DIR = PROJECT_ROOT / "scores" / postfix
BATCH_SIZE = 20000 #500000
BATCH_FILE_FORMAT = 'csv'  # csv удобно для текущей загрузки в HDFS; можно поставить 'parquet'

CPU times: user 100 μs, sys: 0 ns, total: 100 μs
Wall time: 122 μs


In [81]:
%%time
# Все 8 моделей обязательны; загрузка старых июльских/сентябрьских запрещена.
# Модели соответствующей ветки: пути определены в configs/inference_config.json.
# Ноябрьские модели for_november не используются автоматически.
model_paths = INFERENCE_CONFIG['model_paths']
for _key, _spec in model_paths.items():
    _file = PROJECT_ROOT / _spec['path']
    if not _file.is_file():
        raise FileNotFoundError(f'Не найдена октябрьская модель {_key}: {_file}. Сначала выполните train 2.1–2.4')
    if _spec.get('calibrator_path') and not (PROJECT_ROOT / _spec['calibrator_path']).is_file():
        raise FileNotFoundError(f'Нет калибратора {_key}: {_spec["calibrator_path"]}')
models = load_catboost_models(model_paths, project_root=PROJECT_ROOT)
calibrators = load_optional_calibrators(model_paths, project_root=PROJECT_ROOT)
output_columns = model_output_columns(model_paths)

models.keys()

CPU times: user 2.06 s, sys: 199 ms, total: 2.25 s
Wall time: 2.19 s


dict_keys(['pos_loyal', 'pos_noloyal', 'binary_loyal', 'binary_noloyal', 'delta_loyal', 'delta_noloyal', 'cashback_connected', 'loyalty_connection'])

In [82]:
%%time
inference_input_columns = infer_required_input_columns(
    models=models,
    config=INFERENCE_CONFIG,
    id_columns=['cdi_id', 'client_id', 'forecast_month', 'period_start_date', 'offer_scenario_id'],
)

print(f'Колонок для чтения: {len(inference_input_columns)}')
inference_input_columns[:20]

Колонок для чтения: 551
CPU times: user 3.98 ms, sys: 0 ns, total: 3.98 ms
Wall time: 3.73 ms


['accounts_summary__balance_avg',
 'accounts_summary__balance_change_count',
 'accounts_summary__balance_change_lag_3_count',
 'accounts_summary__balance_lag_1_avg',
 'accounts_summary__balance_lag_1_change_count',
 'accounts_summary__balance_lag_1_max',
 'accounts_summary__balance_lag_1_median',
 'accounts_summary__balance_lag_1_min',
 'accounts_summary__balance_lag_3_avg',
 'accounts_summary__balance_lag_3_max',
 'accounts_summary__balance_lag_3_min',
 'accounts_summary__balance_last_lag_3_value',
 'accounts_summary__balance_last_value',
 'accounts_summary__balance_last_value_lag_1_diff',
 'accounts_summary__balance_max',
 'accounts_summary__balance_max_product_lag_1_name',
 'accounts_summary__balance_max_product_lag_3_name',
 'accounts_summary__balance_max_product_name',
 'accounts_summary__balance_median',
 'accounts_summary__balance_min']

In [83]:
import pyarrow.dataset as ds

dataset = ds.dataset(INFERENCE_DATASET_PATH, format="parquet")

available_cols = set(dataset.schema.names)
required_cols = set(inference_input_columns)

missing_cols = sorted(required_cols - available_cols)

print("Колонок в Parquet:", len(available_cols))
print("Запрошено для инференса:", len(required_cols))
print("Отсутствует:", len(missing_cols))
print("Отсутствующие колонки:", missing_cols)

assert {
    "cdi_id",
    "forecast_month",
    "offer_scenario_id"
}.issubset(available_cols), "Отсутствуют обязательные идентификаторы!"

Колонок в Parquet: 1347
Запрошено для инференса: 551
Отсутствует: 21
Отсутствующие колонки: ['active_product2', 'binary_loyal_pred', 'binary_noloyal_pred', 'cashback_expect', 'cashback_pred', 'client_id', 'code_branch', 'code_department', 'delta_loyal_pred', 'delta_noloyal_pred', 'gender', 'loyal_pred', 'period_start_date', 'pos_expect', 'pos_loyal_pred', 'pos_noloyal_pred', 'product_active', 'product_segment', 'segment', 'significant_group', 'time_zone']


In [84]:
missing_set = set(missing_cols)

for model_name, model in models.items():
    missing_model_features = sorted(
        missing_set.intersection(model.feature_names_)
    )

    print(
        f"{model_name}: "
        f"{len(missing_model_features)} отсутствующих признаков"
    )

    if missing_model_features:
        print(missing_model_features)

pos_loyal: 0 отсутствующих признаков
pos_noloyal: 0 отсутствующих признаков
binary_loyal: 0 отсутствующих признаков
binary_noloyal: 0 отсутствующих признаков
delta_loyal: 0 отсутствующих признаков
delta_noloyal: 0 отсутствующих признаков
cashback_connected: 0 отсутствующих признаков
loyalty_connection: 0 отсутствующих признаков


In [86]:
# Собираем категориальные признаки всех 8 моделей
model_cat_features = {
    name: [
        model.feature_names_[i]
        for i in model.get_cat_feature_indices()
    ]
    for name, model in models.items()
}

all_cat_features = sorted(
    set().union(*map(set, model_cat_features.values()))
)

print("Всего уникальных категориальных признаков:", len(all_cat_features))

# Ищем пропущенные значения
for col in all_cat_features:
    if col not in df_test.columns:
        continue

    null_count = int(df_test[col].isna().sum())

    if null_count > 0:
        used_by = [
            name
            for name, features in model_cat_features.items()
            if col in features
        ]

        print(
            f"{col}: "
            f"NULL={null_count}, "
            f"dtype={df_test[col].dtype}, "
            f"модели={used_by}"
        )

Всего уникальных категориальных признаков: 9
accounts_summary__balance_max_product_lag_1_name: NULL=110, dtype=object, модели=['binary_loyal', 'cashback_connected', 'loyalty_connection']
accounts_summary__balance_max_product_lag_3_name: NULL=220, dtype=object, модели=['cashback_connected']
accounts_summary__balance_max_product_name: NULL=55, dtype=object, модели=['binary_noloyal', 'cashback_connected', 'loyalty_connection']
dbo_session__top_platform_name: NULL=275, dtype=object, модели=['binary_loyal', 'binary_noloyal', 'cashback_connected', 'loyalty_connection']
push_communication__top_2_month_push_product_name: NULL=450, dtype=object, модели=['binary_loyal', 'loyalty_connection']
story_communication__top_2_month_click_story_product_lag_1_name: NULL=945, dtype=object, модели=['loyalty_connection']
story_communication__top_2_month_story_product_lag_1_name: NULL=165, dtype=object, модели=['binary_noloyal']
story_communication__top_month_click_story_product_lag_1_name: NULL=835, dtype=ob

In [89]:
%%time

from nbc_project.inference_helpers import iter_parquet_dataset_batches

# Читаем только первые 1000 строк
df_test = next(
    iter_parquet_dataset_batches(
        input_path=INFERENCE_DATASET_PATH,
        columns=inference_input_columns,
        batch_size=1000,
    )
)

print("Строк в тесте:", len(df_test))

# Применяем тот же препроцессинг, что и при полном инференсе
df_test = prepare_inference_frame(
    df_test,
    pos_values=INFERENCE_CONFIG["pos_values"],
    season_base_cols=INFERENCE_CONFIG["season_base_cols"],
    cat_features=INFERENCE_CONFIG["cat_features"],
    numeric_cols=INFERENCE_CONFIG["numeric_string_cols"],
    preprocessing_mode=INFERENCE_CONFIG["preprocessing_mode"],
)

# Прогнозируем всеми восемью моделями
df_test_scored = predict_with_models(
    df_test,
    models=models,
    output_columns=output_columns,
    calibrators=calibrators,
    strict_features=True,
)

print("Скоринг успешно выполнен!")
print("Строк после скоринга:", len(df_test_scored))

pred_cols = [
    col for col in output_columns.values()
    if col in df_test_scored.columns
]

display(
    df_test_scored[
        ["cdi_id", "offer_scenario_id"] + pred_cols
    ].head(5)
)

Строк в тесте: 1000
Скоринг успешно выполнен!
Строк после скоринга: 1000


,cdi_id,offer_scenario_id,pos_loyal_pred,pos_noloyal_pred,binary_loyal_pred,binary_noloyal_pred,delta_loyal_pred,delta_noloyal_pred,cashback_pred,loyal_pred
0,33304488909,fix_0.67_1,17390.832256,7286.978655,0.999290,0.999981,262.203313,191.232216,98.801206,0.031836
1,33304488909,client_random_topn_20,17354.326735,7294.634025,0.999300,0.999980,262.203313,191.232216,104.895869,0.025220
2,33304488909,client_random_topn_17,18234.231320,7353.307503,0.999310,0.999980,263.486023,191.232216,150.331159,0.129151
3,33304488909,fix_3.00_1,17680.116650,7276.476454,0.999323,0.999980,262.203313,191.232216,133.923988,0.020000
4,33304488909,fix_0.72_1,17245.712010,7301.189070,0.999302,0.999981,262.203313,191.232216,96.640818,0.022680


CPU times: user 3min 57s, sys: 21.3 s, total: 4min 18s
Wall time: 3min 58s


In [90]:
from time import perf_counter
from catboost import Pool, CatBoostClassifier

timings = []

for model_name, model in models.items():

    features = list(model.feature_names_)
    cat_indices = list(model.get_cat_feature_indices())

    # Время создания Pool
    t0 = perf_counter()

    pool = Pool(
        df_test[features],
        cat_features=cat_indices,
    )

    pool_sec = perf_counter() - t0

    # Время предсказания
    t0 = perf_counter()

    if isinstance(model, CatBoostClassifier):
        pred = model.predict_proba(pool)
    else:
        pred = model.predict(pool)

    predict_sec = perf_counter() - t0

    timings.append({
        "model": model_name,
        "pool_sec": round(pool_sec, 2),
        "predict_sec": round(predict_sec, 2),
        "total_sec": round(pool_sec + predict_sec, 2),
    })

    print(
        f"{model_name}: "
        f"Pool={pool_sec:.2f} сек, "
        f"Predict={predict_sec:.2f} сек"
    )

    del pool, pred

df_timings = pd.DataFrame(timings)

display(
    df_timings.sort_values(
        "total_sec",
        ascending=False
    )
)

print(
    "Суммарное время:",
    round(df_timings["total_sec"].sum(), 2),
    "сек"
)

pos_loyal: Pool=0.02 сек, Predict=0.00 сек
pos_noloyal: Pool=0.01 сек, Predict=0.00 сек
binary_loyal: Pool=0.03 сек, Predict=0.00 сек
binary_noloyal: Pool=0.01 сек, Predict=0.00 сек
delta_loyal: Pool=0.01 сек, Predict=0.00 сек
delta_noloyal: Pool=0.01 сек, Predict=0.00 сек
cashback_connected: Pool=0.02 сек, Predict=0.00 сек
loyalty_connection: Pool=0.02 сек, Predict=0.00 сек


,model,pool_sec,predict_sec,total_sec
0,pos_loyal,0.02,0.0,0.03
2,binary_loyal,0.03,0.0,0.03
6,cashback_connected,0.02,0.0,0.03
1,pos_noloyal,0.01,0.0,0.02
3,binary_noloyal,0.01,0.0,0.02
7,loyalty_connection,0.02,0.0,0.02
5,delta_noloyal,0.01,0.0,0.01
4,delta_loyal,0.01,0.0,0.01


Суммарное время: 0.17 сек


In [91]:
from time import perf_counter

# 1. Чтение Parquet
t0 = perf_counter()

df_raw = next(
    ih.iter_parquet_dataset_batches(
        input_path=INFERENCE_DATASET_PATH,
        columns=inference_input_columns,
        batch_size=1000,
    )
)

print(
    f"Чтение Parquet: {perf_counter() - t0:.2f} сек",
    flush=True
)

# 2. Препроцессинг
t0 = perf_counter()

df_prepared = ih.prepare_inference_frame(
    df_raw,
    pos_values=INFERENCE_CONFIG["pos_values"],
    season_base_cols=INFERENCE_CONFIG["season_base_cols"],
    cat_features=INFERENCE_CONFIG["cat_features"],
    numeric_cols=INFERENCE_CONFIG["numeric_string_cols"],
    preprocessing_mode=INFERENCE_CONFIG["preprocessing_mode"],
)

print(
    f"Препроцессинг: {perf_counter() - t0:.2f} сек",
    flush=True
)

print("Размер:", df_prepared.shape)

Чтение Parquet: 247.04 сек
Препроцессинг: 0.23 сек
Размер: (1000, 547)


In [92]:
from time import perf_counter

reader = ih.iter_parquet_dataset_batches(
    input_path=INFERENCE_DATASET_PATH,
    columns=inference_input_columns,
    batch_size=10000,
)

try:
    for i in range(1, 4):
        t0 = perf_counter()

        batch = next(reader)

        elapsed = perf_counter() - t0

        print(
            f"Батч {i}: "
            f"{len(batch):,} строк, "
            f"чтение {elapsed:.2f} сек",
            flush=True
        )

        del batch
finally:
    reader.close()

Батч 1: 10,000 строк, чтение 13.81 сек
Батч 2: 10,000 строк, чтение 0.40 сек
Батч 3: 10,000 строк, чтение 0.25 сек


In [ ]:
import pandas as pd

df_model_features = pd.DataFrame([
    {
        "model": name,
        "n_features": len(model.feature_names_),
        "n_cat_features": len(model.get_cat_feature_indices()),
    }
    for name, model in models.items()
])

display(df_model_features)

In [94]:
# Важно: удаляем прошлые scores_part только в персональной директории запуска.
import shutil
if RUN_BATCH_INFERENCE and BATCH_INFERENCE_OUTPUT_DIR.exists():
    shutil.rmtree(BATCH_INFERENCE_OUTPUT_DIR)
if RUN_BATCH_INFERENCE:
    score_part_paths = score_inference_parquet_in_batches(
        input_path=INFERENCE_DATASET_PATH,
        output_dir=BATCH_INFERENCE_OUTPUT_DIR,
        models=models,
        inference_config=INFERENCE_CONFIG,
        output_columns=output_columns,
        calibrators=calibrators,
        columns=inference_input_columns,
        batch_size=BATCH_SIZE,
        score_columns=INFERENCE_CONFIG['score_columns'],
        offer_batch_func=None,
        file_format=BATCH_FILE_FORMAT,
    )

    print(f'Готово part-файлов: {len(score_part_paths)}')

batch=1, rows=20,000, rows_total=20,000, saved=/home/jovyan/work/loyalty_handoff/dynamic/scores/dec_active_base_dynamic_2026_rub_filtered/scores_part_0001.csv
batch=2, rows=20,000, rows_total=40,000, saved=/home/jovyan/work/loyalty_handoff/dynamic/scores/dec_active_base_dynamic_2026_rub_filtered/scores_part_0002.csv
batch=3, rows=20,000, rows_total=60,000, saved=/home/jovyan/work/loyalty_handoff/dynamic/scores/dec_active_base_dynamic_2026_rub_filtered/scores_part_0003.csv
batch=4, rows=20,000, rows_total=80,000, saved=/home/jovyan/work/loyalty_handoff/dynamic/scores/dec_active_base_dynamic_2026_rub_filtered/scores_part_0004.csv
batch=5, rows=20,000, rows_total=100,000, saved=/home/jovyan/work/loyalty_handoff/dynamic/scores/dec_active_base_dynamic_2026_rub_filtered/scores_part_0005.csv
batch=6, rows=20,000, rows_total=120,000, saved=/home/jovyan/work/loyalty_handoff/dynamic/scores/dec_active_base_dynamic_2026_rub_filtered/scores_part_0006.csv
batch=7, rows=20,000, rows_total=140,000, sa

In [97]:
%%time
import pandas as pd
# Лёгкая проверка результата без чтения всех part-файлов в память.
if RUN_BATCH_INFERENCE and BATCH_FILE_FORMAT == 'csv':
    first_part = sorted(BATCH_INFERENCE_OUTPUT_DIR.glob('scores_part_*.csv'))[0]
    display(pd.read_csv(first_part, nrows=10))
elif RUN_BATCH_INFERENCE and BATCH_FILE_FORMAT == 'parquet':
    first_part = sorted(BATCH_INFERENCE_OUTPUT_DIR.glob('scores_part_*.parquet'))[0]
    display(pd.read_parquet(first_part).head())

,cdi_id,offer_scenario_id,pos_loyal_pred,pos_noloyal_pred,binary_loyal_pred,binary_noloyal_pred,delta_loyal_pred,delta_noloyal_pred,cashback_pred,loyal_pred,cashback_expect,pos_expect,batch_num
0,33304488909,fix_0.67_1,17390.832256,7286.978655,0.999290,0.999981,262.203313,191.232216,98.801206,0.031836,3.145462,7608.647672,1
1,33304488909,client_random_topn_20,17354.326735,7294.634025,0.999300,0.999980,262.203313,191.232216,104.895869,0.025220,2.645468,7548.338885,1
2,33304488909,client_random_topn_17,18234.231320,7353.307503,0.999310,0.999980,263.486023,191.232216,150.331159,0.129151,19.415463,8758.592866,1
3,33304488909,fix_3.00_1,17680.116650,7276.476454,0.999323,0.999980,262.203313,191.232216,133.923988,0.020000,2.678480,7484.549258,1
4,33304488909,fix_0.72_1,17245.712010,7301.189070,0.999302,0.999981,262.203313,191.232216,96.640818,0.022680,2.191854,7526.734951,1
5,33304488909,fix_0.61_1,17261.270470,7290.606364,0.999275,0.999981,262.203313,191.232216,86.137418,0.020000,1.722748,7490.019646,1
6,33304488909,fix_0.86_1,18613.430407,7356.744592,0.999316,0.999980,263.769382,191.232216,170.076082,0.050590,8.604186,7926.222795,1
7,33304488909,fix_0.56_1,16845.796968,7261.431856,0.999313,0.999981,262.203313,191.232216,102.207930,0.025220,2.577678,7503.148982,1
8,33304488909,fix_3.75_1,17712.145258,7363.420505,0.999314,0.999980,262.203313,191.232216,136.083654,0.022680,3.086433,7598.133850,1
9,33304488909,client_random_topn_2,17515.411559,7331.502430,0.999312,0.999981,262.203313,191.232216,122.026943,0.031836,3.884883,7655.720117,1


CPU times: user 41.8 ms, sys: 2.66 ms, total: 44.5 ms
Wall time: 43.6 ms


In [98]:
parts = sorted(BATCH_INFERENCE_OUTPUT_DIR.glob("scores_part_*.csv"))

print("Количество файлов:", len(parts))
print("Первый файл:", parts[0].name)
print("Последний файл:", parts[-1].name)
print("Размер последнего файла:", round(parts[-1].stat().st_size / 1024**2, 2), "МБ")

Количество файлов: 1926
Первый файл: scores_part_0001.csv
Последний файл: scores_part_1927.csv
Размер последнего файла: 2.68 МБ


In [99]:
import subprocess

parts = sorted(
    BATCH_INFERENCE_OUTPUT_DIR.glob("scores_part_*.csv")
)

# Проверяем пропуски в нумерации
file_numbers = {
    int(p.stem.split("_")[-1])
    for p in parts
}

missing_numbers = sorted(
    set(range(1, max(file_numbers) + 1)) - file_numbers
)

# Быстрый подсчёт строк через Linux wc
result = subprocess.run(
    ["wc", "-l", *[str(p) for p in parts]],
    capture_output=True,
    text=True,
    check=True,
)

total_lines = int(
    result.stdout.strip().splitlines()[-1].split()[0]
)

# Вычитаем по одной строке заголовка на CSV
total_rows = total_lines - len(parts)

expected_rows = 38_313_825

print("Количество файлов:", len(parts))
print("Пропущенные номера:", missing_numbers)
print("Строк с прогнозами:", f"{total_rows:,}")
print("Ожидаемое количество:", f"{expected_rows:,}")
print("Разница:", total_rows - expected_rows)
print("Количество совпадает:", total_rows == expected_rows)

Количество файлов: 1926
Пропущенные номера: [981]
Строк с прогнозами: 38,313,825
Ожидаемое количество: 38,313,825
Разница: 0
Количество совпадает: True


In [ ]:
%%time
# Старый full-inference оставлен только для маленьких выборок/debug.
# На больших данных RUN_FULL_INFERENCE должен быть False, иначе kernel может упасть.
if RUN_FULL_INFERENCE:
    df_infer = read_partitioned_parquet_folder(INFERENCE_DATASET_PATH)
    df_infer = prepare_inference_frame(
        df_infer,
        pos_values=INFERENCE_CONFIG['pos_values'],
        season_base_cols=INFERENCE_CONFIG['season_base_cols'],
        cat_features=INFERENCE_CONFIG['cat_features'],
        numeric_cols=INFERENCE_CONFIG['numeric_string_cols'],
    )
    df_infer = predict_with_models(
        df_infer,
        models=models,
        output_columns=output_columns,
        calibrators=calibrators,
    )
    if {'cashback_pred', 'loyal_pred'}.issubset(df_infer.columns):
        df_infer['cashback_expect'] = df_infer['cashback_pred'] * df_infer['loyal_pred']

    score_columns = [col for col in INFERENCE_CONFIG['score_columns'] if col in df_infer.columns]
    df_infer[score_columns].to_csv(BATCH_INFERENCE_OUTPUT_DIR / "debug_full_inference.csv", index=False)
    display(df_infer[score_columns].head())


In [100]:
import subprocess

HDFS_SCORE_PATH = (
    f"/warehouse/tablespace/external/hive/"
    f"stg_datamarts.db/temp_uploads/{postfix}"
)

# Не допускаем случайной перезаписи результатов
check = subprocess.run(
    ["hdfs", "dfs", "-test", "-e", HDFS_SCORE_PATH]
)

if check.returncode == 0:
    raise FileExistsError(
        f"Папка уже существует: {HDFS_SCORE_PATH}"
    )

subprocess.run(
    ["hdfs", "dfs", "-mkdir", "-p", HDFS_SCORE_PATH],
    check=True
)

print("HDFS_SCORE_PATH:", HDFS_SCORE_PATH)

def loading_at_file(source_folder):
    return upload_folder_to_hdfs(
        source_folder=source_folder,
        hdfs_path=HDFS_SCORE_PATH,
        namenode_url=INFERENCE_CONFIG["hdfs"]["namenode_url"],
    )

HDFS_SCORE_PATH: /warehouse/tablespace/external/hive/stg_datamarts.db/temp_uploads/dec_active_base_dynamic_2026_rub_filtered


In [ ]:
%%time
import os
SAVE_DF_PATH = os.path.join('/home/jovyan/Artem/NBC/project_nbc_scenarios/export') ## папка для сохранения 

In [ ]:
!pip install hdfs

In [102]:
%%time
from rail_connectors import connect
import rail_connectors

conn_hive = connect(to='DATALAKE')

Successfully login!
CPU times: user 3.1 ms, sys: 4.67 ms, total: 7.78 ms
Wall time: 46.6 ms


In [103]:
%%time
conn_impala = connect(to='IMPALA')

Successfully login!
CPU times: user 4.04 ms, sys: 3.42 ms, total: 7.47 ms
Wall time: 37.2 ms


In [ ]:
BATCH_INFERENCE_OUTPUT_DIR

In [101]:
%%time

# Путь HDFS уже задан выше для эксперимента _rub_filtered.
# Повторно его не переопределяем.

print("Локальная папка:", BATCH_INFERENCE_OUTPUT_DIR)
print("Целевая папка HDFS:", HDFS_SCORE_PATH)

loading_at_file(BATCH_INFERENCE_OUTPUT_DIR)

print("Загрузка CSV в HDFS завершена.")

Локальная папка: /home/jovyan/work/loyalty_handoff/dynamic/scores/dec_active_base_dynamic_2026_rub_filtered
Целевая папка HDFS: /warehouse/tablespace/external/hive/stg_datamarts.db/temp_uploads/dec_active_base_dynamic_2026_rub_filtered
files in hdfs dir = []
Start copying data to HDFS: /warehouse/tablespace/external/hive/stg_datamarts.db/temp_uploads/dec_active_base_dynamic_2026_rub_filtered
File scores_part_0001.csv written successfully; permissions changed to 0777
File scores_part_0002.csv written successfully; permissions changed to 0777
File scores_part_0003.csv written successfully; permissions changed to 0777
File scores_part_0004.csv written successfully; permissions changed to 0777
File scores_part_0005.csv written successfully; permissions changed to 0777
File scores_part_0006.csv written successfully; permissions changed to 0777
File scores_part_0007.csv written successfully; permissions changed to 0777
File scores_part_0008.csv written successfully; permissions changed to 07

In [104]:
%%time

# Название внешней таблицы для текущего эксперимента
EXT_SCORES_TABLE = f"sandbox_crft.mae_ext_nbc_scores_{postfix}"

# Проверяем количество файлов в HDFS
import subprocess

result = subprocess.run(
    ["hdfs", "dfs", "-count", HDFS_SCORE_PATH],
    capture_output=True,
    text=True,
    check=True
)

hdfs_files = int(result.stdout.split()[1])

local_files = len(
    list(BATCH_INFERENCE_OUTPUT_DIR.glob("scores_part_*.csv"))
)

print("Файлов локально:", local_files)
print("Файлов в HDFS:", hdfs_files)

assert hdfs_files == local_files, "Не все файлы загружены в HDFS!"

# Создаём внешнюю таблицу
drop = f"""
DROP TABLE IF EXISTS {EXT_SCORES_TABLE}
"""

create = f"""
CREATE EXTERNAL TABLE {EXT_SCORES_TABLE} (
    cdi_id BIGINT,
    offer_scenario_id STRING,
    pos_loyal_pred DOUBLE,
    pos_noloyal_pred DOUBLE,
    binary_loyal_pred DOUBLE,
    binary_noloyal_pred DOUBLE,
    delta_loyal_pred DOUBLE,
    delta_noloyal_pred DOUBLE,
    cashback_pred DOUBLE,
    loyal_pred DOUBLE,
    cashback_expect DOUBLE,
    pos_expect DOUBLE,
    batch_num INT
)
ROW FORMAT SERDE
    'org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe'
WITH SERDEPROPERTIES (
    'field.delim'=',',
    'serialization.null.format'=''
)
STORED AS INPUTFORMAT
    'org.apache.hadoop.mapred.TextInputFormat'
OUTPUTFORMAT
    'org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat'
LOCATION '{HDFS_SCORE_PATH}'
TBLPROPERTIES (
    'skip.header.line.count'='1'
)
"""

with conn_hive:
    conn_hive.execute(drop)
    conn_hive.execute(create)

print("Внешняя Hive-таблица создана:", EXT_SCORES_TABLE)

Файлов локально: 1926
Файлов в HDFS: 1926


ERROR StatusLogger Unrecognized format specifier [d]
ERROR StatusLogger Unrecognized conversion specifier [d] starting at position 16 in conversion pattern.
ERROR StatusLogger Unrecognized format specifier [thread]
ERROR StatusLogger Unrecognized conversion specifier [thread] starting at position 25 in conversion pattern.
ERROR StatusLogger Unrecognized format specifier [level]
ERROR StatusLogger Unrecognized conversion specifier [level] starting at position 35 in conversion pattern.
ERROR StatusLogger Unrecognized format specifier [logger]
ERROR StatusLogger Unrecognized conversion specifier [logger] starting at position 47 in conversion pattern.
ERROR StatusLogger Unrecognized format specifier [msg]
ERROR StatusLogger Unrecognized conversion specifier [msg] starting at position 54 in conversion pattern.
ERROR StatusLogger Unrecognized format specifier [n]
ERROR StatusLogger Unrecognized conversion specifier [n] starting at position 56 in conversion pattern.
ERROR StatusLogger Unrecog

Внешняя Hive-таблица создана: sandbox_crft.mae_ext_nbc_scores_dec_active_base_dynamic_2026_rub_filtered
CPU times: user 1.54 s, sys: 79.2 ms, total: 1.62 s
Wall time: 20.6 s


In [105]:
%%time

# Обновляем метаданные внешней таблицы
with conn_impala:
    conn_impala.execute(f"""
        INVALIDATE METADATA {EXT_SCORES_TABLE}
    """)

    df_check = conn_impala.fetch(f"""
        SELECT
            COUNT(*) AS cnt_rows,
            COUNT(cdi_id) AS cnt_cdi,
            COUNT(offer_scenario_id) AS cnt_scenarios
        FROM {EXT_SCORES_TABLE}
    """)

print(df_check)

# Проверяем полноту данных
expected_rows = 38_313_825

assert int(df_check["cnt_rows"].iloc[0]) == expected_rows, \
    "Количество строк не совпадает!"

assert int(df_check["cnt_cdi"].iloc[0]) == expected_rows, \
    "Есть строки без cdi_id!"

assert int(df_check["cnt_scenarios"].iloc[0]) == expected_rows, \
    "Есть строки без offer_scenario_id!"

print("Проверки пройдены успешно!")

   cnt_rows   cnt_cdi cnt_scenarios
0  38313825  38313825      38313825
Проверки пройдены успешно!
CPU times: user 249 ms, sys: 22 ms, total: 271 ms
Wall time: 6.05 s


In [106]:
%%time

FINAL_SCORES_TABLE = (
    f"sandbox_crft.mae_nbc_optimizer_scores_{postfix}"
)

drop = f"""
DROP TABLE IF EXISTS {FINAL_SCORES_TABLE}
"""

create = f"""
CREATE TABLE {FINAL_SCORES_TABLE}
STORED AS PARQUET
TBLPROPERTIES ('transactional'='false')
AS
SELECT *
FROM {EXT_SCORES_TABLE}
"""

with conn_impala:
    conn_impala.execute(drop)
    conn_impala.execute(create)

print("Итоговая таблица создана:", FINAL_SCORES_TABLE)

Итоговая таблица создана: sandbox_crft.mae_nbc_optimizer_scores_dec_active_base_dynamic_2026_rub_filtered
CPU times: user 113 ms, sys: 37.3 ms, total: 150 ms
Wall time: 3.45 s


In [107]:
%%time

query = f"""
SELECT
    COUNT(*) AS cnt_rows,
    COUNT(DISTINCT cdi_id) AS cnt_clients,
    COUNT(DISTINCT offer_scenario_id) AS cnt_scenarios,

    SUM(CASE WHEN pos_loyal_pred IS NULL THEN 1 ELSE 0 END) AS null_pos_loyal,
    SUM(CASE WHEN pos_noloyal_pred IS NULL THEN 1 ELSE 0 END) AS null_pos_noloyal,
    SUM(CASE WHEN binary_loyal_pred IS NULL THEN 1 ELSE 0 END) AS null_binary_loyal,
    SUM(CASE WHEN binary_noloyal_pred IS NULL THEN 1 ELSE 0 END) AS null_binary_noloyal,
    SUM(CASE WHEN delta_loyal_pred IS NULL THEN 1 ELSE 0 END) AS null_delta_loyal,
    SUM(CASE WHEN delta_noloyal_pred IS NULL THEN 1 ELSE 0 END) AS null_delta_noloyal,
    SUM(CASE WHEN cashback_pred IS NULL THEN 1 ELSE 0 END) AS null_cashback,
    SUM(CASE WHEN loyal_pred IS NULL THEN 1 ELSE 0 END) AS null_loyal,

    MIN(loyal_pred) AS min_loyal_pred,
    MAX(loyal_pred) AS max_loyal_pred,
    AVG(loyal_pred) AS avg_loyal_pred

FROM {FINAL_SCORES_TABLE}
"""

with conn_impala:
    df_check = conn_impala.fetch(query)

display(df_check)

assert int(df_check["cnt_rows"].iloc[0]) == 38_313_825

null_cols = [c for c in df_check.columns if c.startswith("null_")]
assert (df_check[null_cols].iloc[0] == 0).all(), \
    "Обнаружены NULL в предсказаниях!"

print("Финальная проверка успешно пройдена!")

/home/jovyan/.local/lib/python3.11/site-packages/rail_connectors/connection/dbconnection/_base.py:319: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(lst_df, ignore_index=True)


,cnt_rows,cnt_clients,cnt_scenarios,null_pos_loyal,null_pos_noloyal,null_binary_loyal,null_binary_noloyal,null_delta_loyal,null_delta_noloyal,null_cashback,null_loyal,min_loyal_pred,max_loyal_pred,avg_loyal_pred
0,38313825,696615,55,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.141338


Финальная проверка успешно пройдена!
CPU times: user 161 ms, sys: 121 ms, total: 282 ms
Wall time: 11.9 s


In [ ]:
%%time
with conn_impala:
    for query in [
        f"drop table if exists sandbox_crft.mae_nbc_optimizer_scores_dec_active_base_dynamic_2026",
        f"invalidate metadata sandbox_crft.mae_ext_nbc_scores_dec_active_base_dynamic_2026",
        f"""
        create table sandbox_crft.mae_nbc_optimizer_scores_dec_active_base_dynamic_2026 TBLPROPERTIES ('transactional'='false') as
    	select *
    	from sandbox_crft.mae_ext_nbc_scores_dec_active_base_dynamic_2026""",
        f"drop table if exists sandbox_crft.mae_ext_nbc_scores_dec_active_base_dynamic_2026"
    ]:
        conn_impala.execute(query)

In [ ]:
%%time
# !rm -rf scores/october/infer_oct_pos_parts